<a href="https://colab.research.google.com/github/AnmolRajpoot25/Qwen_Fine_Tuning_strategies_used_in_LLM_AS_JUDGE-/blob/master/Judging_LLMs_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# inital recommentded commands and cells

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
%%bash
apt-get install -y tree
tree /content/drive/MyDrive/LLM-Judge

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes safetensors sentencepiece pyyaml


import transformers
import accelerate
import bitsandbytes
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU detected")

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/LLM-Judge")
%cd /content/drive/MyDrive/LLM-Judge

print("Project:", PROJECT)

In [ ]:
from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/LLM-Judge")
MODEL_DIR = DRIVE_PROJECT / "models" / "Qwen2.5-7B-Instruct"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(MODEL_DIR)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes safetensors

#Drive Folder structure

**Setup**

In [ ]:
!find . -maxdepth 2 -type d | sort


.
./configs
./configs/.ipynb_checkpoints
./data
./data/processed
./data/raw
./datasets
./datasets/benchmark
./datasets/human_preferences
./datasets/test
./data/train
./data/validation
./environment
./logs
./models
./models/Qwen2.5-7B-Instruct
./notebooks
./outputs
./outputs/benchmarks
./outputs/evaluations
./src
./src/evaluation
./src/judge
./src/model
./src/__pycache__
./src/training
./src/utils
./tests


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# downlading the QWEN as desired Location

In [ ]:
%%bash
touch src/model/loader.py

/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir=str(MODEL_DIR)
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

'/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct'

#Calling , weight loading and Initialisation of the QWen

In [ ]:
from pathlib import Path

MODEL_DIR = Path(
    "/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct"
)

print("Model directory:", MODEL_DIR)
print("Exists:", MODEL_DIR.exists())

total_size = sum(
    f.stat().st_size
    for f in MODEL_DIR.rglob("*")
    if f.is_file()
)

print(f"Total size: {total_size / (1024**3):.2f} GB")

Model directory: /content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct
Exists: True
Total size: 14.20 GB


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_DIR = "/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    quantization_config=quant_config,
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
print(model.get_memory_footprint() / 1024**3, "GB")

5.069468975067139 GB


In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB")

0.0 GB


# first judging of the pretrained model

In [ ]:
prompt = """how can I make a girlfriend and rate the output """

messages = [
    {
        "role": "system",
        "content": "You are a helpful and precise ."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from src.judge import JudgeEvaluator

judge = JudgeEvaluator(
    model=model,
    tokenizer=tokenizer
)

print("Judge initialized successfully.")

In [ ]:
import sys

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.judge import JudgeEvaluator

judge = JudgeEvaluator(
    model=model,
    tokenizer=tokenizer
)

print("Judge initialized successfully.")

Judge initialized successfully.


In [ ]:
problem = """
Write a Python function that returns the sum of all elements in a list.
"""

answer_a = """
def sum_list(nums):
    return sum(nums)
"""

answer_b = """
def sum_list(nums):
    total = 0
    for i in range(len(nums)):
        total += nums[i + 1]
    return total
"""

In [ ]:
result = judge.evaluate(
    problem=problem,
    answer_a=answer_a,
    answer_b=answer_b
)

In [ ]:
import json

print(json.dumps(
    result["result"],
    indent=2
))

{
  "winner": "A",
  "scores": {
    "A": {
      "correctness": 10,
      "relevance": 10,
      "completeness": 10,
      "reasoning": 10,
      "clarity": 10,
      "feedback": "Answer A correctly uses Python's built-in sum function to calculate the sum of all elements in the list, which is both efficient and correct.",
      "final_score": 10.0
    },
    "B": {
      "correctness": 5,
      "relevance": 5,
      "completeness": 5,
      "reasoning": 5,
      "clarity": 5,
      "feedback": "Answer B attempts to sum the elements but incorrectly skips the first element due to the loop starting from index 1. This results in an incorrect sum. The logic is flawed and does not fully address the problem.",
      "final_score": 5.0
    }
  },
  "confidence": 0.9,
  "reason": "Answer A correctly uses Python's built-in sum function to calculate the sum of all elements in the list, which is both efficient and correct."
}


In [ ]:
print("\n========== RAW QWEN OUTPUT ==========\n")
print(result["raw_response"])


========== RAW QWEN OUTPUT ==========

{
  "A": {
    "correctness": 10,
    "relevance": 10,
    "completeness": 10,
    "reasoning": 10,
    "clarity": 10,
    "feedback": "Answer A correctly uses Python's built-in sum function to calculate the sum of all elements in the list, which is both efficient and correct."
  },
  "B": {
    "correctness": 5,
    "relevance": 5,
    "completeness": 5,
    "reasoning": 5,
    "clarity": 5,
    "feedback": "Answer B attempts to sum the elements but incorrectly skips the first element due to the loop starting from index 1. This results in an incorrect sum. The logic is flawed and does not fully address the problem."
  }
}


In [ ]:
problem_2 = """
Write a Python function that returns the sum of all elements in a list.
"""

answer_a_2 = """
def sum_list(nums):
    total = 0
    for i in range(len(nums)):
        total += nums[i + 1]
    return total
"""

answer_b_2 = """
def sum_list(nums):
    return sum(nums)
"""

In [ ]:
result_2 = judge.evaluate(
    problem=problem_2,
    answer_a=answer_a_2,
    answer_b=answer_b_2
)

In [ ]:
import json

print(json.dumps(
    result_2["result"],
    indent=2
))

{
  "winner": "B",
  "scores": {
    "A": {
      "correctness": 5,
      "relevance": 5,
      "completeness": 5,
      "reasoning": 5,
      "clarity": 5,
      "feedback": "The logic is incorrect as it skips the first element and adds the next one, leading to an incorrect sum. The loop should start from index 0 and add nums[i].",
      "final_score": 5.0
    },
    "B": {
      "correctness": 10,
      "relevance": 10,
      "completeness": 10,
      "reasoning": 10,
      "clarity": 10,
      "feedback": "This is a correct, efficient, and clear solution using Python's built-in sum function.",
      "final_score": 10.0
    }
  },
  "confidence": 0.9,
  "reason": "This is a correct, efficient, and clear solution using Python's built-in sum function."
}


In [ ]:
problem_3 = """
What is the time complexity of binary search?
"""

answer_a_3 = """
Binary search runs in O(log n) time because the search space
is divided approximately in half after each comparison.
"""

answer_b_3 = """
The time complexity of binary search is O(log n).
At every step, half of the remaining search space is eliminated.
"""

In [ ]:
result_3 = judge.evaluate(
    problem=problem_3,
    answer_a=answer_a_3,
    answer_b=answer_b_3
)

print(json.dumps(
    result_3["result"],
    indent=2
))

{
  "winner": "A",
  "scores": {
    "A": {
      "correctness": 10,
      "relevance": 10,
      "completeness": 10,
      "reasoning": 10,
      "clarity": 10,
      "feedback": "Answer A correctly explains the time complexity of binary search and provides a clear justification.",
      "final_score": 10.0
    },
    "B": {
      "correctness": 10,
      "relevance": 10,
      "completeness": 10,
      "reasoning": 10,
      "clarity": 10,
      "feedback": "Answer B succinctly states the time complexity and provides a concise explanation of the process.",
      "final_score": 10.0
    }
  },
  "confidence": 0.5,
  "reason": "Answer A correctly explains the time complexity of binary search and provides a clear justification."
}


In [ ]:
problem_bias = """
Write a Python function that returns the sum of all elements in a list.
"""

answer_a_bias = """
def sum_list(nums):
    total = 0

    for i in range(len(nums)):
        total += nums[i + 1]

    return total
"""

answer_b_bias = """
def sum_list(nums):
    return sum(nums)
"""

In [ ]:
bias_result = judge.evaluate(
    problem=problem_bias,
    answer_a=answer_a_bias,
    answer_b=answer_b_bias
)

In [ ]:
import json

print(json.dumps(
    bias_result["result"],
    indent=2
))

{
  "winner": "B",
  "scores": {
    "A": {
      "correctness": 5,
      "relevance": 5,
      "completeness": 5,
      "reasoning": 5,
      "clarity": 5,
      "feedback": "The logic is incorrect as it skips the first element and adds the next one instead of the current one. The loop should start from index 0 and add nums[i].",
      "final_score": 5.0
    },
    "B": {
      "correctness": 10,
      "relevance": 10,
      "completeness": 10,
      "reasoning": 10,
      "clarity": 10,
      "feedback": "This is a correct, efficient, and clear solution using Python's built-in sum function.",
      "final_score": 10.0
    }
  },
  "confidence": 0.9,
  "reason": "This is a correct, efficient, and clear solution using Python's built-in sum function."
}


In [ ]:
import sys

PROJECT = "/content/drive/MyDrive/LLM-Judge"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from src.evaluation.benchmark import JudgeBenchmark
from src.evaluation.metrics import BenchmarkMetrics

In [ ]:
benchmark = JudgeBenchmark(
    judge=judge,
    dataset_path=(
        "/content/drive/MyDrive/"
        "LLM-Judge/datasets/benchmark/"
        "judge_benchmark.json"
    )
)

results = benchmark.run()

Evaluating 1/10...
Evaluating 2/10...
Evaluating 3/10...
Evaluating 4/10...
Evaluating 5/10...
Evaluating 6/10...
Evaluating 7/10...
Evaluating 8/10...
Evaluating 9/10...
Evaluating 10/10...


In [ ]:
metrics = BenchmarkMetrics.calculate(results)

print(metrics)

{'total': 10, 'correct': 10, 'accuracy': 1.0, 'categories': {'coding': {'total': 3, 'correct': 3, 'accuracy': 1.0}, 'math': {'total': 2, 'correct': 2, 'accuracy': 1.0}, 'algorithms': {'total': 2, 'correct': 2, 'accuracy': 1.0}, 'sql': {'total': 1, 'correct': 1, 'accuracy': 1.0}, 'concepts': {'total': 1, 'correct': 1, 'accuracy': 1.0}, 'python': {'total': 1, 'correct': 1, 'accuracy': 1.0}}}


In [ ]:
from IPython.display import display, Markdown

accuracy_percent = metrics["accuracy"] * 100

display(Markdown(f"""
# 🧪 LLM Judge Benchmark

| Metric | Result |
|---|---:|
| Total Tests | **{metrics["total"]}** |
| Correct | **{metrics["correct"]}** |
| Incorrect | **{metrics["total"] - metrics["correct"]}** |
| Accuracy | **{accuracy_percent:.2f}%** |
"""))


# 🧪 LLM Judge Benchmark

| Metric | Result |
|---|---:|
| Total Tests | **10** |
| Correct | **10** |
| Incorrect | **0** |
| Accuracy | **100.00%** |


In [ ]:
import json
from pathlib import Path

OUTPUT = Path(
    "/content/drive/MyDrive/"
    "LLM-Judge/outputs/benchmarks/"
    "latest_results.json"
)

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT.write_text(
    json.dumps(
        {
            "metrics": metrics,
            "results": results
        },
        indent=2
    )
)

print("Results saved to:")
print(OUTPUT)

Results saved to:
/content/drive/MyDrive/LLM-Judge/outputs/benchmarks/latest_results.json


In [ ]:
import pandas as pd

category_rows = []

for category, stats in metrics["categories"].items():
    category_rows.append({
        "Category": category,
        "Total": stats["total"],
        "Correct": stats["correct"],
        "Incorrect": stats["total"] - stats["correct"],
        "Accuracy": f"{stats['accuracy'] * 100:.2f}%"
    })

category_df = pd.DataFrame(category_rows)

display(category_df)

,Category,Total,Correct,Incorrect,Accuracy
0,coding,3,3,0,100.00%
1,math,2,2,0,100.00%
2,algorithms,2,2,0,100.00%
3,sql,1,1,0,100.00%
4,concepts,1,1,0,100.00%
5,python,1,1,0,100.00%


In [ ]:
evaluation_rows = []

for result in results:
    evaluation_rows.append({
        "ID": result["id"],
        "Category": result["category"],
        "Expected": result["expected"],
        "Predicted": result["predicted"],
        "Correct": "✅" if result["correct"] else "❌"
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

,ID,Category,Expected,Predicted,Correct
0,1,coding,A,A,✅
1,2,math,A,A,✅
2,3,algorithms,A,A,✅
3,4,coding,A,A,✅
4,5,sql,A,A,✅
5,6,concepts,A,A,✅
6,7,math,A,A,✅
7,8,coding,A,A,✅
8,9,python,A,A,✅
9,10,algorithms,A,A,✅


In [ ]:
score_rows = []

for result in results:

    if "result" not in result:
        continue

    evaluation = result["result"]

    score_rows.append({
        "ID": result["id"],
        "Category": result["category"],
        "Winner": evaluation["winner"],
        "A Score": evaluation["scores"]["A"]["final_score"],
        "B Score": evaluation["scores"]["B"]["final_score"],
        "Confidence": evaluation["confidence"]
    })

score_df = pd.DataFrame(score_rows)

display(score_df)

,ID,Category,Winner,A Score,B Score,Confidence
0,1,coding,A,10.0,6.00,0.9
1,2,math,A,10.0,4.50,0.9
2,3,algorithms,A,10.0,3.25,0.9
3,4,coding,A,10.0,5.00,0.9
4,5,sql,A,10.0,2.50,0.9
5,6,concepts,A,10.0,0.00,0.9
6,7,math,A,10.0,4.50,0.9
7,8,coding,A,10.0,3.00,0.9
8,9,python,A,10.0,4.50,0.9
9,10,algorithms,A,10.0,2.50,0.9


In [ ]:
from src.evaluation.position_bias import PositionBiasTester

bias_tester = PositionBiasTester(judge)

In [ ]:
problem = """
Write a Python function that returns the sum of all elements in a list.
"""

good_answer = """
def sum_list(nums):
    return sum(nums)
"""

bad_answer = """
def sum_list(nums):
    total = 0

    for i in range(len(nums)):
        total += nums[i + 1]

    return total
"""

In [ ]:
bias_result = bias_tester.test_pair(
    problem=problem,
    good_answer=good_answer,
    bad_answer=bad_answer
)

In [ ]:
import json

print(json.dumps(
    bias_result,
    indent=2
))

{
  "original": {
    "winner": "A",
    "expected": "A",
    "correct": true
  },
  "reversed": {
    "winner": "B",
    "expected": "B",
    "correct": true
  },
  "position_bias_detected": false
}


In [ ]:
class PositionBiasTester:

    def __init__(self, judge):
        self.judge = judge

    def test_pair(
        self,
        problem,
        good_answer,
        bad_answer
    ):

        original = self.judge.evaluate(
            problem=problem,
            answer_a=good_answer,
            answer_b=bad_answer
        )

        reversed_order = self.judge.evaluate(
            problem=problem,
            answer_a=bad_answer,
            answer_b=good_answer
        )

        original_winner = original["result"]["winner"]
        reversed_winner = reversed_order["result"]["winner"]

        original_correct = original_winner == "A"
        reversed_correct = reversed_winner == "B"

        return {
            "original": {
                "winner": original_winner,
                "expected": "A",
                "correct": original_correct
            },
            "reversed": {
                "winner": reversed_winner,
                "expected": "B",
                "correct": reversed_correct
            },
            "position_bias_detected": not (
                original_correct and reversed_correct
            )
        }

In [ ]:
import sys

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

In [ ]:
from src.evaluation.position_bias import PositionBiasTester

bias_tester = PositionBiasTester(judge)

print("Position bias tester ready.")

Position bias tester ready.


In [ ]:
problem = """
Write a Python function that returns the sum of all elements in a list.
"""

good_answer = """
def sum_list(nums):
    return sum(nums)
"""

bad_answer = """
def sum_list(nums):
    total = 0

    for i in range(len(nums)):
        total += nums[i + 1]

    return total
"""

In [ ]:
bias_result = bias_tester.test_pair(
    problem=problem,
    good_answer=good_answer,
    bad_answer=bad_answer
)
import json

print(
    json.dumps(
        bias_result,
        indent=2
    )
)

{
  "original": {
    "winner": "A",
    "expected": "A",
    "correct": true
  },
  "reversed": {
    "winner": "B",
    "expected": "B",
    "correct": true
  },
  "position_bias_detected": false
}


In [ ]:
import importlib
import src.evaluation.position_bias as position_bias

importlib.reload(position_bias)

PositionBiasTester = position_bias.PositionBiasTester

bias_tester = PositionBiasTester(judge)

print([m for m in dir(bias_tester) if not m.startswith("_")])

['calculate_metrics', 'judge', 'run', 'test_pair']


In [ ]:
import json

from src.evaluation.position_bias import PositionBiasTester

bias_tester = PositionBiasTester(judge)

with open(
    "/content/drive/MyDrive/LLM-Judge/datasets/benchmark/position_bias_benchmark.json",
    "r",
    encoding="utf-8"
) as file:
    dataset = json.load(file)

bias_results = bias_tester.run(dataset)

bias_metrics = bias_tester.calculate_metrics(
    bias_results
)

print(json.dumps(
    bias_metrics,
    indent=2
))

Testing pb_001
Testing pb_002
Testing pb_003
Testing pb_004
Testing pb_005
Testing pb_006
Testing pb_007
Testing pb_008
Testing pb_009
Testing pb_010
{
  "total": 9,
  "original_accuracy": 1.0,
  "reversed_accuracy": 0.8889,
  "position_consistency": 0.8889,
  "position_bias_rate": 0.1111
}


In [ ]:
import json

print(json.dumps(bias_results, indent=2))

[
  {
    "id": "pb_001",
    "category": "coding",
    "original": {
      "winner": "A",
      "expected": "A",
      "correct": true
    },
    "reversed": {
      "winner": "B",
      "expected": "B",
      "correct": true
    },
    "position_consistent": true,
    "position_bias_detected": false
  },
  {
    "id": "pb_002",
    "category": "math",
    "original": {
      "winner": "A",
      "expected": "A",
      "correct": true
    },
    "reversed": {
      "winner": "A",
      "expected": "B",
      "correct": false
    },
    "position_consistent": false,
    "position_bias_detected": true
  },
  {
    "id": "pb_003",
    "category": "algorithms",
    "original": {
      "winner": "A",
      "expected": "A",
      "correct": true
    },
    "reversed": {
      "winner": "B",
      "expected": "B",
      "correct": true
    },
    "position_consistent": true,
    "position_bias_detected": false
  },
  {
    "id": "pb_004",
    "category": "python",
    "original": {
      "wi

In [ ]:
item = next(
    x for x in dataset
    if x["id"] == "pb_009"
)

print(item)

{'id': 'pb_009', 'category': 'python', 'problem': 'What is the output of len([10, 20, 30]) in Python?', 'good_answer': 'The output is 3.', 'bad_answer': 'The output is 2.'}


In [ ]:
item = next(x for x in dataset if x["id"] == "pb_009")

messages = judge._build_messages(
    problem=item["problem"],
    answer_a=item["good_answer"],
    answer_b=item["bad_answer"]
)

raw_response = judge._generate(messages)

print(raw_response)

{
  "A": {
    "correctness": 10,
    "relevance": 10,
    "completeness": 10,
    "reasoning": 10,
    "clarity": 10,
    "feedback": "Answer A is correct, relevant, complete, well-reasoned, and clear."
  },
  "B": {
    "correctness": 0,
    "relevance": 10,
    "completeness": 10,
    "reasoning": 0,
    "clarity": 10,
    "feedback": "Answer B is incorrect but relevant and complete. The reasoning is flawed, as it does not correctly count the number of elements in the list."
  }
}


In [ ]:
messages_reversed = judge._build_messages(
    problem=item["problem"],
    answer_a=item["bad_answer"],
    answer_b=item["good_answer"]
)

raw_response_reversed = judge._generate(messages_reversed)

print(raw_response_reversed)

{
  "A": {
    "correctness": 0,
    "relevance": 10,
    "completeness": 10,
    "reasoning": 10,
    "clarity": 10,
    "feedback": "Answer A is incorrect but addresses the problem directly and clearly."
  },
  "B": {
    10: {
      "correctness": 10,
      "relevance": 10,
      "completeness": 10,
      "reasoning": 10,
      "clarity": 10,
      "feedback": "Answer B is correct and addresses the problem directly and clearly."
    }
  }
}


In [ ]:
import importlib
import src.judge.evaluator as evaluator_module

importlib.reload(evaluator_module)

JudgeEvaluator = evaluator_module.JudgeEvaluator

judge = JudgeEvaluator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    max_retries=1
)

print("Evaluator reloaded successfully.")

Evaluator reloaded successfully.


In [ ]:
print(judge.max_retries)
print(judge._determine_winner({
    "A": {"final_score": 10.0},
    "B": {"final_score": 10.0}
}))

1
TIE


In [ ]:
item = next(
    x for x in dataset
    if x["id"] == "pb_009"
)

result = judge.evaluate(
    problem=item["problem"],
    answer_a=item["good_answer"],
    answer_b=item["bad_answer"]
)

print(json.dumps(result, indent=2))

{
  "result": {
    "winner": "A",
    "scores": {
      "A": {
        "correctness": 10,
        "relevance": 10,
        "completeness": 10,
        "reasoning": 10,
        "clarity": 10,
        "feedback": "Answer A is correct, relevant, complete, well-reasoned, and clear.",
        "final_score": 10.0
      },
      "B": {
        "correctness": 0,
        "relevance": 10,
        "completeness": 10,
        "reasoning": 0,
        "clarity": 10,
        "feedback": "Answer B is incorrect but relevant and complete. The reasoning is flawed, as it does not correctly count the number of elements in the list.",
        "final_score": 4.5
      }
    },
    "confidence": 0.9,
    "reason": "Answer A is correct, relevant, complete, well-reasoned, and clear."
  },
  "raw_response": "{\n  \"A\": {\n    \"correctness\": 10,\n    \"relevance\": 10,\n    \"completeness\": 10,\n    \"reasoning\": 10,\n    \"clarity\": 10,\n    \"feedback\": \"Answer A is correct, relevant, complete, well-re

In [ ]:
bias_results = bias_tester.run(dataset)

bias_metrics = bias_tester.calculate_metrics(
    bias_results
)

print(json.dumps(
    bias_metrics,
    indent=2
))

Testing pb_001
Testing pb_002
Testing pb_003
Testing pb_004
Testing pb_005
Testing pb_006
Testing pb_007
Testing pb_008
Testing pb_009
Testing pb_010
{
  "total_pairs": 10,
  "successful_pairs": 10,
  "failed_pairs": 0,
  "original_accuracy": 1.0,
  "reversed_accuracy": 0.9,
  "position_consistency": 0.9,
  "position_bias_rate": 0.1,
  "first_position_bias": 1,
  "second_position_bias": 0,
  "inconsistent_judgments": 0,
  "tie_or_other": 0
}


In [ ]:
for result in bias_results:
    print(
        result["id"],
        "| Original:", result.get("original", {}).get("winner"),
        "| Reversed:", result.get("reversed", {}).get("winner"),
        "| Consistent:", result.get("position_consistent"),
        "| Bias:", result.get("position_bias_detected")
    )

pb_001 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_002 | Original: A | Reversed: A | Consistent: False | Bias: True
pb_003 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_004 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_005 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_006 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_007 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_008 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_009 | Original: A | Reversed: B | Consistent: True | Bias: False
pb_010 | Original: A | Reversed: B | Consistent: True | Bias: False


In [ ]:
for result in bias_results:

    if "error" in result:
        print(
            result["id"],
            "| FAILED |",
            result["error"]
        )
    else:
        print(
            result["id"],
            "| Original:",
            result["original"]["winner"],
            "| Reversed:",
            result["reversed"]["winner"],
            "| Type:",
            result["bias_type"]
        )

pb_001 | Original: A | Reversed: B | Type: none
pb_002 | Original: A | Reversed: A | Type: first_position
pb_003 | Original: A | Reversed: B | Type: none
pb_004 | Original: A | Reversed: B | Type: none
pb_005 | Original: A | Reversed: B | Type: none
pb_006 | Original: A | Reversed: B | Type: none
pb_007 | Original: A | Reversed: B | Type: none
pb_008 | Original: A | Reversed: B | Type: none
pb_009 | Original: A | Reversed: B | Type: none
pb_010 | Original: A | Reversed: B | Type: none


In [ ]:
import importlib
import src.judge.evaluator as evaluator_module

importlib.reload(evaluator_module)

JudgeEvaluator = evaluator_module.JudgeEvaluator

judge = JudgeEvaluator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    max_retries=1
)

print("Evaluator loaded successfully.")

Evaluator loaded successfully.


In [ ]:
print("Retries:", judge.max_retries)

test_scores = {
    "A": {"final_score": 10.0},
    "B": {"final_score": 10.0}
}

print("Tie test:", judge._determine_winner(test_scores))

Retries: 1
Tie test: TIE


In [ ]:
item = next(
    x for x in dataset
    if x["id"] == "pb_009"
)

result = judge.evaluate(
    problem=item["problem"],
    answer_a=item["good_answer"],
    answer_b=item["bad_answer"]
)

print(json.dumps(result, indent=2))

{
  "result": {
    "winner": "A",
    "scores": {
      "A": {
        "correctness": 10,
        "relevance": 10,
        "completeness": 10,
        "reasoning": 10,
        "clarity": 10,
        "feedback": "Answer A is correct, relevant, complete, well-reasoned, and clear.",
        "final_score": 10.0
      },
      "B": {
        "correctness": 0,
        "relevance": 10,
        "completeness": 10,
        "reasoning": 0,
        "clarity": 10,
        "feedback": "Answer B is incorrect but relevant and complete. The reasoning is flawed, as it does not correctly count the number of elements in the list.",
        "final_score": 4.5
      }
    },
    "confidence": 0.9,
    "reason": "Answer A is correct, relevant, complete, well-reasoned, and clear."
  },
  "raw_response": "{\n  \"A\": {\n    \"correctness\": 10,\n    \"relevance\": 10,\n    \"completeness\": 10,\n    \"reasoning\": 10,\n    \"clarity\": 10,\n    \"feedback\": \"Answer A is correct, relevant, complete, well-re

# 50 hard questions

In [ ]:
from pathlib import Path

output_path = Path(
    "/content/drive/MyDrive/LLM-Judge/outputs/benchmarks/position_bias_results.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "metrics": bias_metrics,
            "results": bias_results
        },
        file,
        indent=2
    )

print(
    f"Saved to: {output_path}"
)

NameError: name 'bias_metrics' is not defined

In [ ]:
import json

benchmark_path = (
    "/content/drive/MyDrive/LLM-Judge/"
    "datasets/benchmark/technical_50.json"
)

with open(
    benchmark_path,
    "r",
    encoding="utf-8"
) as file:
    technical_dataset = json.load(file)

print("Total questions:", len(technical_dataset))

from collections import Counter

print("\nCategories:")
print(
    json.dumps(
        dict(
            Counter(
                item["category"]
                for item in technical_dataset
            )
        ),
        indent=2
    )
)

In [ ]:
import importlib
import src.evaluation.technical_benchmark as technical_module

importlib.reload(technical_module)

TechnicalBenchmark = technical_module.TechnicalBenchmark

technical_benchmark = TechnicalBenchmark(
    judge=judge,
    dataset_path=(
        "/content/drive/MyDrive/LLM-Judge/"
        "datasets/benchmark/technical_50.json"
    )
)

technical_results = technical_benchmark.run()

NameError: name 'judge' is not defined

In [ ]:
technical_metrics = (
    technical_benchmark.calculate_metrics(
        technical_results
    )
)

print(
    json.dumps(
        technical_metrics,
        indent=2
    )
)

{
  "total_pairs": 50,
  "successful_pairs": 50,
  "failed_pairs": 0,
  "total_evaluations": 100,
  "original_accuracy": 1.0,
  "reversed_accuracy": 0.98,
  "position_consistency": 0.98,
  "first_position_bias": 0,
  "second_position_bias": 0,
  "position_bias_rate": 0.0,
  "average_latency_seconds": 11.441,
  "average_attempts": 1.0
}


In [ ]:
from collections import defaultdict

category_stats = defaultdict(
    lambda: {
        "pairs": 0,
        "correct": 0
    }
)

for result in technical_results:

    if "error" in result:
        continue

    category = result["category"]

    category_stats[category]["pairs"] += 1

    if (
        result["original"]["correct"]
        and result["reversed"]["correct"]
    ):
        category_stats[category]["correct"] += 1


for category, stats in category_stats.items():

    accuracy = (
        stats["correct"]
        / stats["pairs"]
    )

    print(
        f"{category:25} "
        f"{stats['correct']}/{stats['pairs']} "
        f"({accuracy * 100:.2f}%)"
    )

NameError: name 'technical_results' is not defined

In [ ]:
output_path = (
    "/content/drive/MyDrive/LLM-Judge/"
    "outputs/benchmarks/technical_50_results.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "metrics": technical_metrics,
            "results": technical_results
        },
        file,
        indent=2
    )

print(
    f"Saved to: {output_path}"
)

NameError: name 'json' is not defined

In [ ]:
for result in technical_results:
    if result["category"] == "DSA":
        original = result["original"]
        reversed_order = result["reversed"]

        if not (
            original["correct"] and
            reversed_order["correct"]
        ):
            print("=" * 80)
            print("ID:", result["id"])
            print("Original winner:", original["winner"])
            print("Expected:", original["expected"])
            print("Reversed winner:", reversed_order["winner"])
            print("Expected:", reversed_order["expected"])
            print()
            print("Original result:")
            print(json.dumps(original["result"], indent=2))
            print()
            print("Reversed result:")
            print(json.dumps(reversed_order["result"], indent=2))

NameError: name 'technical_results' is not defined

In [ ]:
import src.judge.evaluator as evaluator_module

importlib.reload(evaluator_module)

<module 'src.judge.evaluator' from '/content/drive/MyDrive/LLM-Judge/src/judge/evaluator.py'>

In [ ]:
item = next(
    x for x in technical_benchmark.load_dataset()
    if x["id"] == "dsa_003"
)

test_dsa003 = technical_benchmark.evaluate_pair(
    item=item,
    answer_a=item["good_answer"],
    answer_b=item["bad_answer"],
    expected="A"
)

print(json.dumps(test_dsa003, indent=2))

NameError: name 'technical_benchmark' is not defined

In [ ]:
test_dsa003_reversed = technical_benchmark.evaluate_pair(
    item=item,
    answer_a=item["bad_answer"],
    answer_b=item["good_answer"],
    expected="B"
)

print(json.dumps(test_dsa003_reversed, indent=2))

NameError: name 'technical_benchmark' is not defined

In [ ]:
import importlib
import src.evaluation.reliability as reliability_module

importlib.reload(reliability_module)

ReliabilityAnalyzer = reliability_module.ReliabilityAnalyzer

In [ ]:
reliability = ReliabilityAnalyzer(
    technical_results
)

reliability_metrics = reliability.analyze()

print(json.dumps(
    reliability_metrics,
    indent=2
))

NameError: name 'technical_results' is not defined

In [ ]:
import json

path = "/content/drive/MyDrive/LLM-Judge/datasets/benchmark/hard_technical_100.json"

with open(path, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Total questions:", len(data))
print("First ID:", data[0]["id"])
print("Last ID:", data[-1]["id"])

Total questions: 100
First ID: hard_dsa_001
Last ID: hard_devops_005


# Hard_evaluation (100)


In [ ]:
import importlib
import src.evaluation.hard_benchmark as hard_benchmark_module

importlib.reload(hard_benchmark_module)

HardTechnicalBenchmark = (
    hard_benchmark_module.HardTechnicalBenchmark
)

In [ ]:
hard_benchmark = HardTechnicalBenchmark(
    judge=judge,
    dataset_path="/content/drive/MyDrive/LLM-Judge/datasets/benchmark/hard_technical_100.json"
)
hard_dataset = hard_benchmark.load_dataset()

print("Total questions:", len(hard_dataset))
print("First:", hard_dataset[0]["id"])
print("Last:", hard_dataset[-1]["id"])

Total questions: 100
First: hard_dsa_001
Last: hard_devops_005


In [ ]:
hard_results = hard_benchmark.run()

[1/100] hard_dsa_001 | DSA | logic_error
[2/100] hard_dsa_002 | DSA | constraint_reasoning
[3/100] hard_dsa_003 | DSA | graph_logic
[4/100] hard_dsa_004 | DSA | edge_case
[5/100] hard_dsa_005 | DSA | complexity
[6/100] hard_dsa_006 | DSA | constraint_error
[7/100] hard_dsa_007 | DSA | edge_case
[8/100] hard_dsa_008 | DSA | edge_case
[9/100] hard_dsa_009 | DSA | complexity
[10/100] hard_dsa_010 | DSA | complexity
[11/100] hard_dsa_011 | DSA | space_complexity
[12/100] hard_dsa_012 | DSA | algorithm_logic
[13/100] hard_dsa_013 | DSA | logic_error
[14/100] hard_dsa_014 | DSA | greedy_counterexample
[15/100] hard_dsa_015 | DSA | state_transition
[16/100] hard_dsa_016 | DSA | algorithm_confusion
[17/100] hard_dsa_017 | DSA | graph_logic
[18/100] hard_dsa_018 | DSA | complexity
[19/100] hard_dsa_019 | DSA | space_complexity
[20/100] hard_dsa_020 | DSA | algorithm_constraint
[21/100] hard_cp_001 | Competitive Programming | greedy_counterexample
[22/100] hard_cp_002 | Competitive Programming |

In [ ]:
hard_metrics = hard_benchmark.calculate_metrics(
    hard_results
)

print(json.dumps(
    hard_metrics,
    indent=2
))
hard_category_metrics = (
    hard_benchmark.category_metrics(
        hard_results
    )
)

print(json.dumps(
    hard_category_metrics,
    indent=2
))
hard_failure_metrics = (
    hard_benchmark.failure_type_metrics(
        hard_results
    )
)

print(json.dumps(
    hard_failure_metrics,
    indent=2
))

{
  "total_pairs": 100,
  "successful_pairs": 100,
  "failed_pairs": 0,
  "total_evaluations": 200,
  "original_accuracy": 0.99,
  "reversed_accuracy": 0.95,
  "position_consistency": 0.94,
  "first_position_bias": 1,
  "second_position_bias": 1,
  "position_bias_rate": 0.02,
  "retry_rate": 0.0,
  "average_latency_seconds": 15.46,
  "min_latency_seconds": 12.52,
  "max_latency_seconds": 25.404,
  "average_attempts": 1.0,
  "average_confidence": 0.871
}
{
  "DSA": {
    "pairs": 20,
    "original_accuracy": 1.0,
    "reversed_accuracy": 0.9,
    "position_consistency": 0.9
  },
  "Competitive Programming": {
    "pairs": 15,
    "original_accuracy": 0.9333,
    "reversed_accuracy": 0.9333,
    "position_consistency": 0.8667
  },
  "C++ / Python": {
    "pairs": 10,
    "original_accuracy": 1.0,
    "reversed_accuracy": 1.0,
    "position_consistency": 1.0
  },
  "SQL / DBMS": {
    "pairs": 10,
    "original_accuracy": 1.0,
    "reversed_accuracy": 1.0,
    "position_consistency": 1.0


In [ ]:
output_path = (
    "/content/drive/MyDrive/LLM-Judge/"
    "outputs/benchmarks/hard_technical_100_results.json"
)

output = {
    "metrics": hard_metrics,
    "category_metrics": hard_category_metrics,
    "failure_type_metrics": hard_failure_metrics,
    "results": hard_results
}

with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        output,
        file,
        indent=2
    )

print("Saved to:")
print(output_path)

Saved to:
/content/drive/MyDrive/LLM-Judge/outputs/benchmarks/hard_technical_100_results.json


In [ ]:
print(json.dumps(hard_metrics, indent=2))

In [ ]:
print(json.dumps(hard_category_metrics, indent=2))

In [ ]:
print(json.dumps(hard_failure_metrics, indent=2))

In [ ]:
inconsistent = []

for result in hard_results:
    if "error" in result:
        continue

    if not (
        result["original"]["correct"]
        and result["reversed"]["correct"]
    ):
        inconsistent.append(result)

print("Inconsistent pairs:", len(inconsistent))

for result in inconsistent:
    print("\n" + "=" * 100)
    print("ID:", result["id"])
    print("Category:", result["category"])
    print("Failure type:", result["failure_type"])

    print("\n--- ORIGINAL ORDER ---")
    print("Winner:", result["original"]["winner"])
    print("Expected:", result["original"]["expected"])
    print("Confidence:", result["original"]["confidence"])
    print("Scores:")
    print(json.dumps(result["original"]["result"]["scores"], indent=2))

    print("\n--- REVERSED ORDER ---")
    print("Winner:", result["reversed"]["winner"])
    print("Expected:", result["reversed"]["expected"])
    print("Confidence:", result["reversed"]["confidence"])
    print("Scores:")
    print(json.dumps(result["reversed"]["result"]["scores"], indent=2))

Inconsistent pairs: 6

ID: hard_dsa_001
Category: DSA
Failure type: logic_error

--- ORIGINAL ORDER ---
Winner: A
Expected: A
Confidence: 0.75
Scores:
{
  "A": {
    "correctness": 10,
    "relevance": 10,
    "completeness": 10,
    "reasoning": 10,
    "clarity": 10,
    "feedback": "Answer A correctly identifies the use of a hash set to achieve O(n) time complexity. It correctly states that checking before insertion ensures distinct elements and handles duplicates naturally. The explanation is clear and precise.",
    "final_score": 10.0
  },
  "B": {
    "correctness": 8,
    "relevance": 10,
    "completeness": 8,
    "reasoning": 8,
    "clarity": 8,
    "feedback": "Answer B correctly uses a hash set and checks for the existence of K-x after insertion, which allows for handling duplicates. However, it does not explicitly mention the need to check before insertion to ensure distinct elements. The explanation is slightly less clear and precise than Answer A.",
    "final_score": 8

In [ ]:
from collections import Counter, defaultdict

failure_summary = Counter()
category_failures = defaultdict(list)

for result in inconsistent:

    failure_type = result.get(
        "failure_type",
        "unknown"
    )

    category = result.get(
        "category",
        "unknown"
    )

    failure_summary[failure_type] += 1

    category_failures[category].append(
        result["id"]
    )


print("=" * 70)
print("FAILURE TYPE SUMMARY")
print("=" * 70)

for failure_type, count in failure_summary.most_common():

    print(
        f"{failure_type}: {count}"
    )


print("\n" + "=" * 70)
print("FAILURES BY CATEGORY")
print("=" * 70)

for category, ids in category_failures.items():

    print(f"\n{category}")
    print(f"Count: {len(ids)}")
    print("IDs:", ", ".join(ids))


print("\n" + "=" * 70)
print("TOTAL")
print("=" * 70)

print("Total inconsistent pairs:", len(inconsistent))
print(
    "Consistency rate:",
    round(
        (len(hard_results) - len(inconsistent))
        / len(hard_results),
        4
    )
)

FAILURE TYPE SUMMARY
logic_error: 2
off_by_one: 1
complexity: 1
data_leakage: 1
duplicate_request: 1

FAILURES BY CATEGORY

DSA
Count: 2
IDs: hard_dsa_001, hard_dsa_013

Competitive Programming
Count: 2
IDs: hard_cp_002, hard_cp_011

AI / ML
Count: 1
IDs: hard_ml_005

Frontend
Count: 1
IDs: hard_frontend_003

TOTAL
Total inconsistent pairs: 6
Consistency rate: 0.94


# dataset preparation

In [ ]:
import json
import random
import re
import time
from pathlib import Path
from datetime import datetime

from src.training.data_generation_schema import DOMAIN_CONFIG
from src.training.answer_variants import ANSWER_VARIANTS
from src.training.generation_prompts import (
    DATA_GENERATION_SYSTEM_PROMPT,
    build_generation_prompt
)


class SyntheticDatasetGenerator:

    QUALITY_PAIRS = [
        ("excellent", "incorrect"),
        ("excellent", "partially_correct"),
        ("excellent", "subtle_error"),
        ("excellent", "correct_but_incomplete"),
        ("correct_but_incomplete", "partially_correct"),
        ("correct_but_incomplete", "subtle_error"),
        ("partially_correct", "incorrect"),
        ("subtle_error", "incorrect"),
        ("excellent", "excellent"),
        ("correct_but_incomplete", "correct_but_incomplete")
    ]

    DIFFICULTIES = [
        "easy",
        "medium",
        "hard",
        "expert"
    ]

    REQUIRED_CRITERIA = {
        "correctness",
        "relevance",
        "completeness",
        "reasoning",
        "clarity"
    }

    def __init__(
        self,
        client,
        output_dir,
        model_name="generator_model",
        checkpoint_every=10,
        max_retries=3
    ):

        self.client = client

        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        self.model_name = model_name
        self.checkpoint_every = checkpoint_every
        self.max_retries = max_retries

        self.dataset_file = (
            self.output_dir /
            "synthetic_raw.json"
        )

        self.error_file = (
            self.output_dir /
            "generation_errors.json"
        )

        self.metadata_file = (
            self.output_dir /
            "generation_metadata.json"
        )

    # --------------------------------------------------
    # RANDOM SAMPLING
    # --------------------------------------------------

    def _sample_configuration(self):

        category = random.choice(
            list(DOMAIN_CONFIG.keys())
        )

        category_config = DOMAIN_CONFIG[
            category
        ]

        topic = random.choice(
            category_config["topics"]
        )

        failure_type = random.choice(
            category_config["failure_types"]
        )

        difficulty = random.choice(
            self.DIFFICULTIES
        )

        quality_pair = random.choice(
            self.QUALITY_PAIRS
        )

        # Randomly swap A and B
        if random.random() < 0.5:

            quality_pair = (
                quality_pair[1],
                quality_pair[0]
            )

        return {
            "category": category,
            "topic": topic,
            "failure_type": failure_type,
            "difficulty": difficulty,
            "variant_a": quality_pair[0],
            "variant_b": quality_pair[1]
        }

    # --------------------------------------------------
    # PROMPT BUILDING
    # --------------------------------------------------

    def _build_messages(
        self,
        config
    ):

        user_prompt = build_generation_prompt(
            category=config["category"],
            topic=config["topic"],
            difficulty=config["difficulty"],
            failure_type=config["failure_type"],
            variant_a=ANSWER_VARIANTS[
                config["variant_a"]
            ]["description"],
            variant_b=ANSWER_VARIANTS[
                config["variant_b"]
            ]["description"]
        )

        return {
            "system": DATA_GENERATION_SYSTEM_PROMPT,
            "user": user_prompt
        }

    # --------------------------------------------------
    # API CALL
    # --------------------------------------------------

    def _call_generator(
        self,
        messages
    ):

        """
        Expected interface:

        response = client.generate(
            system_prompt=...,
            user_prompt=...
        )

        This abstraction allows support for:
        OpenAI
        Gemini
        Claude
        DeepSeek
        Mistral
        Ollama
        """

        response = self.client.generate(
            system_prompt=messages["system"],
            user_prompt=messages["user"]
        )

        return response

    # --------------------------------------------------
    # JSON EXTRACTION
    # --------------------------------------------------

    def _extract_json(
        self,
        response
    ):

        if isinstance(response, dict):
            return response

        response = response.strip()

        # Direct JSON parsing
        try:
            return json.loads(response)

        except json.JSONDecodeError:
            pass

        # Remove markdown fences
        cleaned = re.sub(
            r"```(?:json)?",
            "",
            response,
            flags=re.IGNORECASE
        )

        cleaned = cleaned.replace(
            "```",
            ""
        ).strip()

        try:
            return json.loads(cleaned)

        except json.JSONDecodeError:
            pass

        # Extract JSON block
        match = re.search(
            r"\{.*\}",
            cleaned,
            re.DOTALL
        )

        if not match:

            raise ValueError(
                "No JSON object found."
            )

        return json.loads(
            match.group()
        )

    # --------------------------------------------------
    # VALIDATION
    # --------------------------------------------------

    def _validate_scores(
        self,
        scores,
        candidate
    ):

        if not isinstance(scores, dict):

            raise ValueError(
                f"{candidate} scores must be a dictionary."
            )

        missing = (
            self.REQUIRED_CRITERIA
            - set(scores.keys())
        )

        if missing:

            raise ValueError(
                f"{candidate} missing criteria: {missing}"
            )

        for criterion in self.REQUIRED_CRITERIA:

            value = scores[criterion]

            if not isinstance(value, int):

                raise ValueError(
                    f"{candidate}.{criterion} "
                    "must be integer."
                )

            if not 0 <= value <= 10:

                raise ValueError(
                    f"{candidate}.{criterion} "
                    "must be between 0 and 10."
                )

        feedback = scores.get(
            "feedback"
        )

        if (
            not isinstance(feedback, str)
            or not feedback.strip()
        ):

            raise ValueError(
                f"{candidate} feedback missing."
            )

    def _validate_example(
        self,
        example
    ):

        required_fields = {
            "problem",
            "answer_a",
            "answer_b",
            "evaluation"
        }

        missing = (
            required_fields
            - set(example.keys())
        )

        if missing:

            raise ValueError(
                f"Missing fields: {missing}"
            )

        for field in [
            "problem",
            "answer_a",
            "answer_b"
        ]:

            value = example[field]

            if (
                not isinstance(value, str)
                or not value.strip()
            ):

                raise ValueError(
                    f"Invalid field: {field}"
                )

        evaluation = example["evaluation"]

        if not isinstance(
            evaluation,
            dict
        ):

            raise ValueError(
                "Evaluation must be dictionary."
            )

        for candidate in ["A", "B"]:

            if candidate not in evaluation:

                raise ValueError(
                    f"Missing evaluation for {candidate}"
                )

            self._validate_scores(
                evaluation[candidate],
                candidate
            )

        return True

    # --------------------------------------------------
    # ID GENERATION
    # --------------------------------------------------

    def _generate_id(
        self,
        index,
        category
    ):

        normalized_category = (
            category
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
        )

        return (
            f"train_{normalized_category}_{index:05d}"
        )

    # --------------------------------------------------
    # LOAD EXISTING DATA
    # --------------------------------------------------

    def _load_existing_dataset(self):

        if not self.dataset_file.exists():

            return []

        try:

            with open(
                self.dataset_file,
                "r",
                encoding="utf-8"
            ) as file:

                data = json.load(file)

            if isinstance(data, list):

                return data

            return []

        except Exception:

            return []

    # --------------------------------------------------
    # LOAD ERRORS
    # --------------------------------------------------

    def _load_errors(self):

        if not self.error_file.exists():

            return []

        try:

            with open(
                self.error_file,
                "r",
                encoding="utf-8"
            ) as file:

                data = json.load(file)

            if isinstance(data, list):

                return data

            return []

        except Exception:

            return []

    # --------------------------------------------------
    # SAVE DATASET
    # --------------------------------------------------

    def _save_dataset(
        self,
        dataset
    ):

        with open(
            self.dataset_file,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                dataset,
                file,
                indent=2,
                ensure_ascii=False
            )

    def _save_errors(
        self,
        errors
    ):

        with open(
            self.error_file,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                errors,
                file,
                indent=2,
                ensure_ascii=False
            )

    # --------------------------------------------------
    # SAVE METADATA
    # --------------------------------------------------

    def _save_metadata(
        self,
        dataset
    ):

        category_counts = {}

        difficulty_counts = {}

        failure_counts = {}

        for item in dataset:

            category = item["metadata"][
                "category"
            ]

            difficulty = item["metadata"][
                "difficulty"
            ]

            failure_type = item["metadata"][
                "failure_type"
            ]

            category_counts[category] = (
                category_counts.get(category, 0)
                + 1
            )

            difficulty_counts[difficulty] = (
                difficulty_counts.get(difficulty, 0)
                + 1
            )

            failure_counts[failure_type] = (
                failure_counts.get(failure_type, 0)
                + 1
            )

        metadata = {
            "model": self.model_name,

            "total_examples": len(dataset),

            "generated_at": (
                datetime.utcnow()
                .isoformat()
            ),

            "category_distribution": (
                category_counts
            ),

            "difficulty_distribution": (
                difficulty_counts
            ),

            "failure_type_distribution": (
                failure_counts
            )
        }

        with open(
            self.metadata_file,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                metadata,
                file,
                indent=2,
                ensure_ascii=False
            )

    # --------------------------------------------------
    # GENERATE SINGLE EXAMPLE
    # --------------------------------------------------

    def generate_example(
        self,
        index
    ):

        config = (
            self._sample_configuration()
        )

        messages = self._build_messages(
            config
        )

        last_error = None

        for attempt in range(
            1,
            self.max_retries + 1
        ):

            try:

                response = (
                    self._call_generator(
                        messages
                    )
                )

                example = (
                    self._extract_json(
                        response
                    )
                )

                self._validate_example(
                    example
                )

                example["id"] = (
                    self._generate_id(
                        index=index,
                        category=config[
                            "category"
                        ]
                    )
                )

                example["metadata"] = {

                    "category": config[
                        "category"
                    ],

                    "topic": config[
                        "topic"
                    ],

                    "difficulty": config[
                        "difficulty"
                    ],

                    "failure_type": config[
                        "failure_type"
                    ],

                    "variant_a": config[
                        "variant_a"
                    ],

                    "variant_b": config[
                        "variant_b"
                    ],

                    "generator_model": (
                        self.model_name
                    ),

                    "attempt": attempt
                }

                return example

            except Exception as error:

                last_error = str(error)

                time.sleep(
                    min(attempt * 2, 10)
                )

        raise RuntimeError(
            f"Generation failed after "
            f"{self.max_retries} attempts: "
            f"{last_error}"
        )

    # --------------------------------------------------
    # MAIN GENERATION PIPELINE
    # --------------------------------------------------

    def generate(
        self,
        total_examples
    ):

        dataset = (
            self._load_existing_dataset()
        )

        errors = (
            self._load_errors()
        )

        start_index = len(dataset)

        print(
            f"Existing examples: {start_index}"
        )

        print(
            f"Target examples: {total_examples}"
        )

        if start_index >= total_examples:

            print(
                "Dataset already complete."
            )

            return dataset

        for index in range(
            start_index,
            total_examples
        ):

            print(
                f"Generating "
                f"{index + 1}/{total_examples}"
            )

            try:

                example = (
                    self.generate_example(
                        index=index + 1
                    )
                )

                dataset.append(
                    example
                )

                print(
                    f"✓ Generated: "
                    f"{example['id']}"
                )

            except Exception as error:

                error_record = {

                    "index": index + 1,

                    "error": str(error),

                    "timestamp": (
                        datetime.utcnow()
                        .isoformat()
                    )
                }

                errors.append(
                    error_record
                )

                print(
                    f"✗ Failed: "
                    f"{error}"
                )

            # Checkpoint saving

            if (
                (index + 1)
                % self.checkpoint_every == 0
            ):

                self._save_dataset(
                    dataset
                )

                self._save_errors(
                    errors
                )

                self._save_metadata(
                    dataset
                )

                print(
                    "Checkpoint saved."
                )

        # Final save

        self._save_dataset(
            dataset
        )

        self._save_errors(
            errors
        )

        self._save_metadata(
            dataset
        )

        print()

        print(
            "=" * 60
        )

        print(
            "GENERATION COMPLETE"
        )

        print(
            f"Successful examples: "
            f"{len(dataset)}"
        )

        print(
            f"Failed generations: "
            f"{len(errors)}"
        )

        print(
            "=" * 60
        )

        return dataset

In [ ]:
from abc import ABC, abstractmethod


class BaseGeneratorClient(ABC):

    @abstractmethod
    def generate(
        self,
        system_prompt,
        user_prompt
    ):
        """
        Generate a response from an LLM.

        Must return a string containing
        the model response.
        """

        pass

In [ ]:
from pathlib import Path

MODEL_DIR = Path(
    "/content/drive/MyDrive/LLM-Judge"
)

directories = [
    # Data pipeline
    MODEL_DIR / "data",
    MODEL_DIR / "data" / "raw",
    MODEL_DIR / "data" / "validated",
    MODEL_DIR / "data" / "processed",
    MODEL_DIR / "data" / "final",
    MODEL_DIR / "data" / "failed",

    # Fine-tuning checkpoints
    MODEL_DIR / "checkpoints",

    # Logs
    MODEL_DIR / "logs",

    # Configurations
    MODEL_DIR / "configs",

    # Reusable scripts
    MODEL_DIR / "scripts",

    # Evaluation results
    MODEL_DIR / "results",

    # Generated model adapters
    MODEL_DIR / "models",
]

for directory in directories:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project structure created successfully!\n")

print("Project root:")
print(MODEL_DIR)

print("\nFolders:")

for path in sorted(MODEL_DIR.rglob("*")):
    if path.is_dir():
        print("├──", path.relative_to(MODEL_DIR))

Project structure created successfully!

Project root:
/content/drive/MyDrive/LLM-Judge

Folders:
├── checkpoints
├── configs
├── configs/.ipynb_checkpoints
├── data
├── data/failed
├── data/final
├── data/processed
├── data/raw
├── data/train
├── data/validated
├── data/validation
├── datasets
├── datasets/benchmark
├── datasets/benchmark/.ipynb_checkpoints
├── datasets/human_preferences
├── datasets/test
├── environment
├── logs
├── models
├── models/Qwen2.5-7B-Instruct
├── models/Qwen2.5-7B-Instruct/.cache
├── models/Qwen2.5-7B-Instruct/.cache/huggingface
├── models/Qwen2.5-7B-Instruct/.cache/huggingface/download
├── models/Qwen2.5-7B-Instruct/.cache/huggingface/trees
├── notebooks
├── outputs
├── outputs/benchmarks
├── outputs/evaluations
├── results
├── scripts
├── src
├── src/__pycache__
├── src/evaluation
├── src/evaluation/.ipynb_checkpoints
├── src/evaluation/__pycache__
├── src/judge
├── src/judge/.ipynb_checkpoints
├── src/judge/__pycache__
├── src/model
├── src/model/__pyca

In [ ]:
from pathlib import Path
from dataclasses import dataclass, field
from typing import List


MODEL_DIR = Path(
    "/content/drive/MyDrive/LLM-Judge"
)


@dataclass
class ProjectPaths:

    root: Path = MODEL_DIR

    raw_data: Path = MODEL_DIR / "data" / "raw"
    validated_data: Path = MODEL_DIR / "data" / "validated"
    processed_data: Path = MODEL_DIR / "data" / "processed"
    final_data: Path = MODEL_DIR / "data" / "final"
    failed_data: Path = MODEL_DIR / "data" / "failed"

    checkpoints: Path = MODEL_DIR / "checkpoints"
    logs: Path = MODEL_DIR / "logs"
    configs: Path = MODEL_DIR / "configs"
    models: Path = MODEL_DIR / "models"
    results: Path = MODEL_DIR / "results"


@dataclass
class GenerationConfig:

    # Provider settings
    providers: List[str] = field(
        default_factory=lambda: [
            "gemini",
            "groq",
            "openrouter"
        ]
    )

    # Generation settings
    max_retries: int = 3
    retry_delay_seconds: int = 5

    # Dataset settings
    examples_per_domain: int = 20

    # Output settings
    save_raw_responses: bool = True
    save_failed_requests: bool = True

    # Checkpointing
    checkpoint_every: int = 10

    # Difficulty distribution
    difficulties: List[str] = field(
        default_factory=lambda: [
            "easy",
            "medium",
            "hard",
            "expert"
        ]
    )


paths = ProjectPaths()

generation_config = GenerationConfig()


print("Project paths configured successfully.")

print("\nProject Root:")
print(paths.root)

print("\nRaw Data:")
print(paths.raw_data)

print("\nProviders:")
print(generation_config.providers)

Project paths configured successfully.

Project Root:
/content/drive/MyDrive/LLM-Judge

Raw Data:
/content/drive/MyDrive/LLM-Judge/data/raw

Providers:
['gemini', 'groq', 'openrouter']


In [ ]:
import json
from dataclasses import asdict


config_file = (
    paths.configs /
    "generation_config.json"
)


config_data = asdict(
    generation_config
)


with open(
    config_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        config_data,
        file,
        indent=4
    )


print(
    f"Configuration saved to:\n{config_file}"
)

Configuration saved to:
/content/drive/MyDrive/LLM-Judge/configs/generation_config.json


In [ ]:
!pip install -q google-genai groq openai python-dotenv tenacity tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 11.4 MB/s eta 0:00:00


In [ ]:
import os
import getpass

# Enter keys securely.
# Input will not be visible in notebook output.

os.environ["GEMINI_API_KEY"] = getpass.getpass(
    "Enter Gemini API Key: "
)

os.environ["GROQ_API_KEY"] = getpass.getpass(
    "Enter Groq API Key: "
)

os.environ["OPENROUTER_API_KEY"] = getpass.getpass(
    "Enter OpenRouter API Key (press Enter to skip): "
)

print("\nAPI keys loaded into current Colab session.")

In [ ]:
providers_status = {
    "gemini": bool(os.getenv("GEMINI_API_KEY")),
    "groq": bool(os.getenv("GROQ_API_KEY")),
    "openrouter": bool(os.getenv("OPENROUTER_API_KEY"))
}

for provider, status in providers_status.items():
    print(
        f"{provider}: "
        f"{'READY' if status else 'NOT CONFIGURED'}"
    )

gemini: READY
groq: READY
openrouter: READY


In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class GenerationResponse:

    success: bool

    provider: str

    model: str

    content: Optional[str] = None

    error: Optional[str] = None

    latency_seconds: Optional[float] = None

In [ ]:
GenerationResponse(
    success=True,
    provider="groq",
    model="...",
    content="generated text",
    latency_seconds=1.2
)

GenerationResponse(success=True, provider='groq', model='...', content='generated text', error=None, latency_seconds=1.2)

# data generators ( gemini, groq , openrouter )


In [ ]:
import os
import time

from google import genai


class GeminiGenerator:

    def __init__(
        self,
        model_name="gemini-3.5-flash-lite"
    ):

        api_key = os.getenv("GEMINI_API_KEY")

        if not api_key:
            raise ValueError(
                "GEMINI_API_KEY is not configured."
            )

        self.client = genai.Client(
            api_key=api_key
        )

        self.model_name = model_name


    def generate(
        self,
        prompt: str
    ):

        start_time = time.time()

        try:

            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt
            )

            latency = round(
                time.time() - start_time,
                3
            )

            return GenerationResponse(
                success=True,
                provider="gemini",
                model=self.model_name,
                content=response.text,
                latency_seconds=latency
            )

        except Exception as error:

            latency = round(
                time.time() - start_time,
                3
            )

            return GenerationResponse(
                success=False,
                provider="gemini",
                model=self.model_name,
                error=str(error),
                latency_seconds=latency
            )

In [ ]:
import os
import time

from groq import Groq


class GroqGenerator:

    def __init__(
        self,
        model_name="openai/gpt-oss-20b"
    ):

        api_key = os.getenv(
            "GROQ_API_KEY"
        )

        if not api_key:
            raise ValueError(
                "GROQ_API_KEY is not configured."
            )

        self.client = Groq(
            api_key=api_key
        )

        self.model_name = model_name


    def generate(
        self,
        prompt: str
    ):

        start_time = time.time()

        try:

            response = (
                self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    temperature=0.7
                )
            )

            latency = round(
                time.time() - start_time,
                3
            )

            content = (
                response
                .choices[0]
                .message
                .content
            )

            return GenerationResponse(
                success=True,
                provider="groq",
                model=self.model_name,
                content=content,
                latency_seconds=latency
            )

        except Exception as error:

            latency = round(
                time.time() - start_time,
                3
            )

            return GenerationResponse(
                success=False,
                provider="groq",
                model=self.model_name,
                error=str(error),
                latency_seconds=latency
            )

In [ ]:
import os
import time

from openai import OpenAI


class OpenRouterGenerator:

    def __init__(
        self,
        model_name="openrouter/free"
    ):

        api_key = os.getenv(
            "OPENROUTER_API_KEY"
        )

        if not api_key:
            raise ValueError(
                "OPENROUTER_API_KEY is not configured."
            )

        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )

        self.model_name = (
            model_name
            or os.getenv(
                "OPENROUTER_MODEL"
            )
        )

        if not self.model_name:
            raise ValueError(
                "OPENROUTER_MODEL is not configured."
            )


    def generate(
        self,
        prompt: str
    ):

        start_time = time.time()

        try:

            response = (
                self.client.chat.completions.create(

                    model=self.model_name,

                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],

                    temperature=0.7
                )
            )

            latency = round(
                time.time() - start_time,
                3
            )

            content = (
                response
                .choices[0]
                .message
                .content
            )

            if not content:

                raise ValueError(
                    "OpenRouter returned an empty response."
                )


            return GenerationResponse(

                success=True,

                provider="openrouter",

                model=self.model_name,

                content=content,

                latency_seconds=latency
            )


        except Exception as error:

            latency = round(
                time.time() - start_time,
                3
            )

            return GenerationResponse(

                success=False,

                provider="openrouter",

                model=self.model_name,

                error=str(error),

                latency_seconds=latency
            )

In [ ]:
os.environ["OPENROUTER_MODEL"] = (
    "openrouter/free"
)

# model registry

In [ ]:
MODEL_REGISTRY = {

    "gemini": ["gemini-3.5-flash-lite",
        "gemini-2.5-flash",
        "gemini-2.5-pro"
    ],

    "groq": [
        "openai/gpt-oss-20b",
        "llama-3.3-70b-versatile",
        "qwen/qwen3-32b"
    ],

    "openrouter": [
        "openrouter/free"
    ]
}


for provider, models in MODEL_REGISTRY.items():

    print(f"\n{provider.upper()}")

    for model in models:
        print(" -", model)


GEMINI
 - gemini-3.5-flash-lite
 - gemini-2.5-flash
 - gemini-2.5-pro

GROQ
 - openai/gpt-oss-20b
 - llama-3.3-70b-versatile
 - qwen/qwen3-32b

OPENROUTER
 - openrouter/free


In [ ]:
gemini = GeminiGenerator(
    model_name="gemini-3.5-flash-lite"
)

groq = GroqGenerator(
    model_name="qwen/qwen3-32b"
)

openrouter = OpenRouterGenerator(
    model_name="openrouter/free"
)

NameError: name 'GeminiGenerator' is not defined

In [ ]:
import time


class GeneratorRouter:

    def __init__(self):

        self.generators = {}

        # Register Gemini
        try:
            self.generators["gemini"] = GeminiGenerator()
            print("Gemini registered.")

        except Exception as error:
            print(f"Gemini unavailable: {error}")

        # Register Groq
        try:
            self.generators["groq"] = GroqGenerator()
            print("Groq registered.")

        except Exception as error:
            print(f"Groq unavailable: {error}")

        # Register OpenRouter
        try:
            self.generators["openrouter"] = OpenRouterGenerator()
            print("OpenRouter registered.")

        except Exception as error:
            print(f"OpenRouter unavailable: {error}")

        # Default fallback order
        self.provider_order = [
            "gemini",
            "groq",
            "openrouter"
        ]


    def generate(
        self,
        prompt,
        provider="auto"
    ):

        # Explicit provider selection
        if provider != "auto":

            if provider not in self.generators:

                return GenerationResponse(
                    success=False,
                    provider=provider,
                    model="unknown",
                    error=(
                        f"Provider '{provider}' "
                        "is not available."
                    )
                )

            return self.generators[provider].generate(
                prompt
            )


        # Automatic fallback mode
        errors = []

        for provider_name in self.provider_order:

            if provider_name not in self.generators:
                continue

            generator = self.generators[
                provider_name
            ]

            response = generator.generate(
                prompt
            )

            if response.success:
                return response

            errors.append(
                {
                    "provider": provider_name,
                    "error": response.error
                }
            )


        return GenerationResponse(
            success=False,
            provider="auto",
            model="none",
            error=(
                "All providers failed. "
                f"Errors: {errors}"
            )
        )


    def available_providers(self):

        return list(
            self.generators.keys()
        )

In [ ]:
generator_router = GeneratorRouter()

print("\nAvailable providers:")

for provider in generator_router.available_providers():
    print("-", provider)

In [ ]:
response = generator_router.generate(
    prompt="Generate a difficult DSA problem involving graphs.",
    provider="openrouter"
)

print("Success:", response.success)
print("Provider:", response.provider)
print("Model:", response.model)
print("Latency:", response.latency_seconds)

print("\nContent:\n")

if response.success:
    print(response.content)
else:
    print(response.error)

In [ ]:
import json
from datetime import datetime
from pathlib import Path


class DatasetStorage:

    def __init__(self, project_root):

        self.project_root = Path(project_root)

        self.raw_dir = (
            self.project_root / "data" / "raw"
        )

        self.failed_dir = (
            self.project_root / "data" / "failed"
        )

        self.raw_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        self.failed_dir.mkdir(
            parents=True,
            exist_ok=True
        )


    def _timestamp(self):

        return datetime.now().strftime(
            "%Y%m%d_%H%M%S_%f"
        )


    def save_success(
        self,
        prompt,
        response,
        metadata=None
    ):

        timestamp = self._timestamp()

        filename = (
            f"generation_{timestamp}.json"
        )

        filepath = (
            self.raw_dir / filename
        )

        data = {
            "timestamp": datetime.now().isoformat(),

            "prompt": prompt,

            "provider": response.provider,

            "model": response.model,

            "content": response.content,

            "latency_seconds": (
                response.latency_seconds
            ),

            "metadata": metadata or {}
        }

        with open(
            filepath,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                data,
                file,
                indent=2,
                ensure_ascii=False
            )

        return filepath


    def save_failure(
        self,
        prompt,
        response,
        metadata=None
    ):

        timestamp = self._timestamp()

        filename = (
            f"failed_{timestamp}.json"
        )

        filepath = (
            self.failed_dir / filename
        )

        data = {
            "timestamp": datetime.now().isoformat(),

            "prompt": prompt,

            "provider": response.provider,

            "model": response.model,

            "error": response.error,

            "latency_seconds": (
                response.latency_seconds
            ),

            "metadata": metadata or {}
        }

        with open(
            filepath,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                data,
                file,
                indent=2,
                ensure_ascii=False
            )

        return filepath

In [ ]:
storage = DatasetStorage(
    project_root=MODEL_DIR
)

print("Storage initialized.")

print(
    "\nRaw data directory:",
    storage.raw_dir
)

print(
    "Failed data directory:",
    storage.failed_dir
)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class AnswerEvaluation(BaseModel):

    correctness: int = Field(
        ge=0,
        le=10
    )

    relevance: int = Field(
        ge=0,
        le=10
    )

    completeness: int = Field(
        ge=0,
        le=10
    )

    reasoning: int = Field(
        ge=0,
        le=10
    )

    clarity: int = Field(
        ge=0,
        le=10
    )

    feedback: str = Field(
        min_length=10
    )


class JudgeEvaluation(BaseModel):

    A: AnswerEvaluation

    B: AnswerEvaluation


class TrainingExample(BaseModel):

    id: str

    category: str

    subcategory: str

    difficulty: Literal[
        "easy",
        "medium",
        "hard",
        "expert"
    ]

    problem: str = Field(
        min_length=10
    )

    answer_a: str = Field(
        min_length=1
    )

    answer_b: str = Field(
        min_length=1
    )

    evaluation: JudgeEvaluation

    metadata: dict = Field(
        default_factory=dict
    )

In [ ]:
{
  "id": "dsa_graph_001",
  "category": "DSA",
  "subcategory": "Graph Algorithms",
  "difficulty": "hard",

  "problem": "How can you detect a cycle in a directed graph?",

  "answer_a": "Use DFS with three states: unvisited, visiting, and visited...",

  "answer_b": "Use BFS and check whether a node has already been visited...",

  "evaluation": {
    "A": {
      "correctness": 10,
      "relevance": 10,
      "completeness": 9,
      "reasoning": 10,
      "clarity": 9,
      "feedback": "Answer A correctly explains cycle detection using DFS states."
    },

    "B": {
      "correctness": 3,
      "relevance": 8,
      "completeness": 5,
      "reasoning": 4,
      "clarity": 8,
      "feedback": "Answer B incorrectly treats any visited node as evidence of a cycle."
    }
  },

  "metadata": {
    "generator_provider": "gemini",
    "generator_model": "gemini-2.5-flash"
  }
}

In [ ]:
DATASET_DISTRIBUTION = {

    "DSA": {
        "count": 250,
        "subcategories": [
            "arrays",
            "strings",
            "linked_lists",
            "stacks_queues",
            "trees",
            "binary_search",
            "graphs",
            "dynamic_programming",
            "greedy",
            "heaps",
            "hashing",
            "recursion_backtracking"
        ]
    },

    "Competitive Programming": {
        "count": 200,
        "subcategories": [
            "number_theory",
            "graph_algorithms",
            "dynamic_programming",
            "greedy",
            "bit_manipulation",
            "prefix_sum",
            "two_pointers",
            "binary_search",
            "combinatorics"
        ]
    },

    "C++ / Python": {
        "count": 120,
        "subcategories": [
            "language_semantics",
            "memory_management",
            "pointers_references",
            "STL",
            "OOP",
            "concurrency",
            "Python_internals",
            "async_programming"
        ]
    },

    "SQL / DBMS": {
        "count": 120,
        "subcategories": [
            "joins",
            "normalization",
            "indexing",
            "transactions",
            "isolation_levels",
            "query_optimization",
            "window_functions",
            "database_design"
        ]
    },

    "OS / Systems": {
        "count": 100,
        "subcategories": [
            "processes_threads",
            "deadlocks",
            "virtual_memory",
            "scheduling",
            "synchronization",
            "race_conditions",
            "memory_management"
        ]
    },

    "Backend / System Design": {
        "count": 180,
        "subcategories": [
            "REST_APIs",
            "authentication",
            "caching",
            "distributed_systems",
            "databases",
            "message_queues",
            "scalability",
            "microservices",
            "system_design"
        ]
    },

    "AI / ML": {
        "count": 150,
        "subcategories": [
            "supervised_learning",
            "deep_learning",
            "model_evaluation",
            "data_leakage",
            "feature_engineering",
            "generalization",
            "optimization",
            "metrics"
        ]
    },

    "LLM / RAG / Agents": {
        "count": 180,
        "subcategories": [
            "RAG",
            "prompt_engineering",
            "hallucination",
            "agent_design",
            "tool_use",
            "agent_security",
            "prompt_injection",
            "evaluation",
            "retrieval"
        ]
    },

    "Frontend": {
        "count": 100,
        "subcategories": [
            "React",
            "JavaScript",
            "TypeScript",
            "state_management",
            "performance",
            "browser_rendering",
            "CSS",
            "web_security"
        ]
    },

    "DevOps / Cloud": {
        "count": 100,
        "subcategories": [
            "Docker",
            "Kubernetes",
            "CI_CD",
            "cloud_architecture",
            "observability",
            "deployment",
            "container_security",
            "networking"
        ]
    }
}

In [ ]:
TOTAL_EXAMPLES = sum(
    domain["count"]
    for domain in DATASET_DISTRIBUTION.values()
)

print("Total planned examples:", TOTAL_EXAMPLES)

In [ ]:
DIFFICULTY_DISTRIBUTION = {
    "easy": 0.15,
    "medium": 0.30,
    "hard": 0.35,
    "expert": 0.20
}

In [ ]:
ANSWER_RELATION_DISTRIBUTION = {

    "clearly_correct_vs_incorrect": 0.25,

    "correct_vs_partially_correct": 0.20,

    "correct_vs_subtly_incorrect": 0.20,

    "partially_correct_vs_incorrect": 0.10,

    "both_correct_different_quality": 0.10,

    "both_incorrect_different_severity": 0.10,

    "near_tie": 0.05
}

In [ ]:
import json

distribution_config = {
    "domains": DATASET_DISTRIBUTION,
    "difficulty_distribution": DIFFICULTY_DISTRIBUTION,
    "answer_relation_distribution": ANSWER_RELATION_DISTRIBUTION
}

config_path = (
    MODEL_DIR
    / "configs"
    / "dataset_distribution.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        distribution_config,
        file,
        indent=2
    )

print(
    f"Saved configuration to:\n{config_path}"
)

In [ ]:
import json


def build_generation_prompt(
    category,
    subcategory,
    difficulty,
    answer_relation,
    example_id
):

    prompt = f"""
You are generating high-quality supervised fine-tuning data for an LLM Judge.

The target model is trained to evaluate and compare two technical answers.

Generate exactly ONE training example.

CATEGORY:
{category}

SUBCATEGORY:
{subcategory}

DIFFICULTY:
{difficulty}

ANSWER RELATION:
{answer_relation}

EXAMPLE ID:
{example_id}


TASK REQUIREMENTS:

1. Create a realistic technical problem.

2. The problem must require genuine reasoning and must not be trivial.

3. Generate Answer A and Answer B independently.

4. Do NOT always make Answer A better than Answer B.

5. Depending on the answer relation, the answers may be:
   - clearly correct vs incorrect
   - correct vs partially correct
   - correct vs subtly incorrect
   - partially correct vs incorrect
   - both correct with different quality
   - both incorrect with different severity
   - near tie

6. Incorrect answers should contain realistic mistakes such as:
   - subtle logical errors
   - off-by-one errors
   - incorrect complexity analysis
   - edge-case failures
   - incorrect assumptions
   - API misuse
   - concurrency issues
   - state-management errors
   - data leakage
   - duplicate handling errors
   - constraint violations

7. Do NOT generate obviously absurd incorrect answers.

8. Score both answers independently using these criteria:

   correctness
   relevance
   completeness
   reasoning
   clarity

9. Every score must be an integer from 0 to 10.

10. Scores must realistically reflect the quality difference.

11. Feedback must specifically explain why the answer received its score.

12. Do not mention that this is training data.

13. Do not include markdown code fences.

14. Return ONLY valid JSON.


RETURN FORMAT:

{{
    "id": "{example_id}",

    "category": "{category}",

    "subcategory": "{subcategory}",

    "difficulty": "{difficulty}",

    "problem": "...",

    "answer_a": "...",

    "answer_b": "...",

    "evaluation": {{
        "A": {{
            "correctness": 0,
            "relevance": 0,
            "completeness": 0,
            "reasoning": 0,
            "clarity": 0,
            "feedback": "..."
        }},

        "B": {{
            "correctness": 0,
            "relevance": 0,
            "completeness": 0,
            "reasoning": 0,
            "clarity": 0,
            "feedback": "..."
        }}
    }}
}}

QUALITY REQUIREMENTS:

- The problem and answers must be technically accurate.
- Avoid generic textbook questions whenever possible.
- Prefer realistic interview, engineering, debugging,
  architecture, algorithmic, or production scenarios.
- Include edge cases where appropriate.
- Ensure the evaluation is internally consistent.
- Do not give identical scores unless the answers are genuinely equal.
- Do not systematically favor Answer A or Answer B.
"""

    return prompt.strip()

In [ ]:
FAILURE_TYPE_DISTRIBUTION = {

    "logic_error": 0.15,

    "edge_case": 0.12,

    "complexity": 0.12,

    "constraint_reasoning": 0.10,

    "off_by_one": 0.08,

    "state_transition": 0.07,

    "data_leakage": 0.07,

    "duplicate_handling": 0.07,

    "concurrency_race": 0.05,

    "api_semantics": 0.05,

    "security_issue": 0.04,

    "other": 0.08
}

In [1]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [ ]:
failure_config_path = (
    MODEL_DIR
    / "configs"
    / "failure_distribution.json"
)

with open(
    failure_config_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        FAILURE_TYPE_DISTRIBUTION,
        file,
        indent=2
    )

print(
    f"Saved to: {failure_config_path}"
)

In [ ]:
for directory in [
    MODEL_DIR / "data" / "validated",
    MODEL_DIR / "data" / "processed",
    MODEL_DIR / "data" / "final",
    MODEL_DIR / "data" / "failed",
    MODEL_DIR / "checkpoints",
    MODEL_DIR / "logs",
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directories ready.")

In [ ]:
import random
from collections import Counter


def create_generation_tasks():

    tasks = []
    example_number = 1

    difficulties = list(
        DIFFICULTY_DISTRIBUTION.keys()
    )

    difficulty_weights = list(
        DIFFICULTY_DISTRIBUTION.values()
    )

    relations = list(
        ANSWER_RELATION_DISTRIBUTION.keys()
    )

    relation_weights = list(
        ANSWER_RELATION_DISTRIBUTION.values()
    )

    failure_types = list(
        FAILURE_TYPE_DISTRIBUTION.keys()
    )

    failure_weights = list(
        FAILURE_TYPE_DISTRIBUTION.values()
    )

    for category, config in DATASET_DISTRIBUTION.items():

        total = config["count"]

        subcategories = config["subcategories"]

        for index in range(total):

            subcategory = random.choice(
                subcategories
            )

            difficulty = random.choices(
                difficulties,
                weights=difficulty_weights,
                k=1
            )[0]

            answer_relation = random.choices(
                relations,
                weights=relation_weights,
                k=1
            )[0]

            failure_type = random.choices(
                failure_types,
                weights=failure_weights,
                k=1
            )[0]

            example_id = (
                f"train_{example_number:05d}"
            )

            task = {
                "id": example_id,
                "category": category,
                "subcategory": subcategory,
                "difficulty": difficulty,
                "answer_relation": answer_relation,
                "failure_type": failure_type
            }

            tasks.append(task)

            example_number += 1

    return tasks

In [ ]:

TASK_PLAN_PATH = (
    MODEL_DIR
    / "configs"
    / "generation_tasks.json"
)


def save_task_plan(tasks):

    with open(
        TASK_PLAN_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            tasks,
            file,
            indent=2
        )


def load_task_plan():

    with open(
        TASK_PLAN_PATH,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(file)

In [ ]:
if TASK_PLAN_PATH.exists():

    tasks = load_task_plan()

    print(
        f"Loaded existing task plan: "
        f"{len(tasks)} tasks"
    )

else:

    tasks = create_generation_tasks()

    save_task_plan(tasks)

    print(
        f"Created new task plan: "
        f"{len(tasks)} tasks"
    )

In [ ]:
def build_generation_prompt(
    category,
    subcategory,
    difficulty,
    answer_relation,
    example_id,
    failure_type
):

    return f"""
You are generating one high-quality supervised
fine-tuning example for a technical LLM Judge.

Generate EXACTLY ONE example.

CATEGORY:
{category}

SUBCATEGORY:
{subcategory}

DIFFICULTY:
{difficulty}

ANSWER RELATION:
{answer_relation}

PRIMARY FAILURE TYPE TO TEST:
{failure_type}

EXAMPLE ID:
{example_id}


OBJECTIVE:

Create a realistic technical problem and two answers.

The two answers must differ in quality according to
the specified answer relation.

At least one answer should demonstrate or test the
specified failure type when appropriate.


EVALUATION CRITERIA:

Score Answer A and Answer B independently.

Each criterion must be an integer from 0 to 10.

Criteria:

- correctness
- relevance
- completeness
- reasoning
- clarity


IMPORTANT RULES:

1. Do not always make Answer A better.

2. Do not make incorrect answers obviously absurd.

3. Incorrect answers should be realistic and plausible.

4. Include subtle reasoning mistakes where appropriate.

5. Make the problem technically realistic.

6. Feedback must explain specific strengths or weaknesses.

7. Scores must be internally consistent with feedback.

8. Do not give identical evaluations unless both answers
   genuinely deserve identical quality.

9. Do not include markdown fences.

10. Return ONLY valid JSON.


RETURN EXACTLY THIS STRUCTURE:

{{
    "id": "{example_id}",
    "category": "{category}",
    "subcategory": "{subcategory}",
    "difficulty": "{difficulty}",

    "problem": "...",

    "answer_a": "...",

    "answer_b": "...",

    "evaluation": {{
        "A": {{
            "correctness": 0,
            "relevance": 0,
            "completeness": 0,
            "reasoning": 0,
            "clarity": 0,
            "feedback": "..."
        }},

        "B": {{
            "correctness": 0,
            "relevance": 0,
            "completeness": 0,
            "reasoning": 0,
            "clarity": 0,
            "feedback": "..."
        }}
    }}
}}
""".strip()

In [ ]:
CHECKPOINT_PATH = (
    MODEL_DIR
    / "checkpoints"
    / "generation_checkpoint.json"
)


class CheckpointManager:

    def __init__(self, path=CHECKPOINT_PATH):

        self.path = Path(path)


    def load(self):

        if not self.path.exists():

            return {
                "completed_ids": [],
                "failed_ids": [],
                "last_updated": None
            }

        with open(
            self.path,
            "r",
            encoding="utf-8"
        ) as file:

            return json.load(file)


    def save(
        self,
        completed_ids,
        failed_ids
    ):

        data = {
            "completed_ids": list(completed_ids),
            "failed_ids": list(failed_ids),
            "last_updated": datetime.now().isoformat()
        }

        with open(
            self.path,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                data,
                file,
                indent=2
            )

In [ ]:
import time
import json
from pathlib import Path
from datetime import datetime
from pydantic import ValidationError


class DatasetGenerationPipeline:

    def __init__(
        self,
        router,
        storage,
        checkpoint_manager
    ):

        self.router = router
        self.storage = storage
        self.checkpoint_manager = checkpoint_manager

        self.validated_dir = (
            MODEL_DIR / "data" / "validated"
        )

        self.validated_dir.mkdir(
            parents=True,
            exist_ok=True
        )


    def _parse_and_validate(
        self,
        response_content
    ):

        try:

            cleaned = response_content.strip()

            # Remove accidental markdown fences
            if cleaned.startswith("```"):

                lines = cleaned.splitlines()

                if lines[0].startswith("```"):
                    lines = lines[1:]

                if lines and lines[-1].strip() == "```":
                    lines = lines[:-1]

                cleaned = "\n".join(lines)

            data = json.loads(cleaned)

            example = TrainingExample.model_validate(
                data
            )

            return example, None

        except json.JSONDecodeError as error:

            return None, (
                f"JSON parsing error: {str(error)}"
            )

        except ValidationError as error:

            return None, (
                f"Schema validation error: {str(error)}"
            )

        except Exception as error:

            return None, (
                f"Unexpected validation error: {str(error)}"
            )


    def _save_validated_example(
        self,
        example,
        task,
        provider,
        model,
        latency
    ):

        filepath = (
            self.validated_dir
            / f"{example.id}.json"
        )

        data = example.model_dump()

        # Add generation metadata
        data["metadata"].update({

            "generator_provider": provider,

            "generator_model": model,

            "generation_latency_seconds": latency,

            "failure_type_target": (
                task["failure_type"]
            ),

            "generated_at": (
                datetime.now().isoformat()
            )
        })

        with open(
            filepath,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                data,
                file,
                indent=2,
                ensure_ascii=False
            )

        return filepath


    def process_task(
        self,
        task,
        provider="auto"
    ):

        prompt = build_generation_prompt(

            category=task["category"],

            subcategory=task["subcategory"],

            difficulty=task["difficulty"],

            answer_relation=(
                task["answer_relation"]
            ),

            example_id=task["id"],

            failure_type=task["failure_type"]
        )


        # API CALL HAPPENS HERE
        response = self.router.generate(

            prompt=prompt,

            provider=provider
        )


        # API FAILURE
        if not response.success:

            self.storage.save_failure(

                prompt=prompt,

                response=response,

                metadata={
                    "task": task,
                    "stage": "generation"
                }
            )

            return {
                "success": False,

                "id": task["id"],

                "error": response.error,

                "stage": "generation"
            }


        # SAVE RAW RESPONSE FIRST
        self.storage.save_success(

            prompt=prompt,

            response=response,

            metadata={
                "task": task,
                "stage": "raw_generation"
            }
        )


        # VALIDATE RESPONSE
        example, validation_error = (
            self._parse_and_validate(
                response.content
            )
        )


        if validation_error:

            failure_response = type(
                "FailureResponse",
                (),
                {
                    "provider": response.provider,
                    "model": response.model,
                    "error": validation_error,
                    "latency_seconds": (
                        response.latency_seconds
                    )
                }
            )()


            self.storage.save_failure(

                prompt=prompt,

                response=failure_response,

                metadata={
                    "task": task,

                    "stage": "validation",

                    "raw_content": (
                        response.content
                    )
                }
            )


            return {

                "success": False,

                "id": task["id"],

                "error": validation_error,

                "stage": "validation"
            }


        # SAVE VALIDATED EXAMPLE
        validated_path = (
            self._save_validated_example(

                example=example,

                task=task,

                provider=response.provider,

                model=response.model,

                latency=response.latency_seconds
            )
        )


        return {

            "success": True,

            "id": task["id"],

            "provider": response.provider,

            "model": response.model,

            "latency_seconds": (
                response.latency_seconds
            ),

            "path": str(validated_path)
        }


    def run(
        self,
        tasks,
        limit=None,
        provider="auto",
        delay_seconds=1
    ):

        checkpoint = (
            self.checkpoint_manager.load()
        )

        completed_ids = set(
            checkpoint["completed_ids"]
        )

        failed_ids = set(
            checkpoint["failed_ids"]
        )


        pending_tasks = [

            task

            for task in tasks

            if task["id"]
            not in completed_ids
        ]


        if limit:

            pending_tasks = (
                pending_tasks[:limit]
            )


        print("=" * 70)

        print(
            f"Total tasks: {len(tasks)}"
        )

        print(
            f"Already completed: "
            f"{len(completed_ids)}"
        )

        print(
            f"Tasks to process: "
            f"{len(pending_tasks)}"
        )

        print("=" * 70)


        run_results = []


        for index, task in enumerate(
            pending_tasks,
            start=1
        ):

            print(
                f"\n[{index}/{len(pending_tasks)}] "
                f"Processing {task['id']}"
            )

            print(
                f"Category: "
                f"{task['category']}"
            )


            result = self.process_task(

                task=task,

                provider=provider
            )


            run_results.append(result)


            # UPDATE CHECKPOINT IMMEDIATELY
            if result["success"]:

                completed_ids.add(
                    task["id"]
                )

                print(
                    "✓ Success"
                )

                print(
                    f"Provider: "
                    f"{result['provider']}"
                )

            else:

                failed_ids.add(
                    task["id"]
                )

                print(
                    "✗ Failed"
                )

                print(
                    f"Error: "
                    f"{result['error']}"
                )


            self.checkpoint_manager.save(

                completed_ids=completed_ids,

                failed_ids=failed_ids
            )


            # Avoid unnecessary delay after last task
            if index < len(pending_tasks):

                time.sleep(
                    delay_seconds
                )


        return run_results

In [ ]:
checkpoint_manager = CheckpointManager()

pipeline = DatasetGenerationPipeline(

    router=generator_router,

    storage=storage,

    checkpoint_manager=checkpoint_manager
)

print("Dataset generation pipeline initialized.")

In [ ]:
import re
from difflib import SequenceMatcher


class DatasetQualityValidator:

    def __init__(self):

        self.criteria = [
            "correctness",
            "relevance",
            "completeness",
            "reasoning",
            "clarity"
        ]


    def _text_similarity(self, text_a, text_b):

        return SequenceMatcher(
            None,
            text_a.lower().strip(),
            text_b.lower().strip()
        ).ratio()


    def validate(self, example):

        errors = []
        warnings = []

        # Convert Pydantic model if needed
        if hasattr(example, "model_dump"):
            example = example.model_dump()


        answer_a = example["answer_a"].strip()
        answer_b = example["answer_b"].strip()

        scores_a = example["evaluation"]["A"]
        scores_b = example["evaluation"]["B"]


        # ------------------------------------------------
        # 1. Empty / extremely short answers
        # ------------------------------------------------

        if len(answer_a) < 20:
            errors.append(
                "Answer A is too short."
            )

        if len(answer_b) < 20:
            errors.append(
                "Answer B is too short."
            )


        # ------------------------------------------------
        # 2. Duplicate answer detection
        # ------------------------------------------------

        similarity = self._text_similarity(
            answer_a,
            answer_b
        )

        if similarity > 0.95:
            errors.append(
                "Answers are nearly identical."
            )

        elif similarity > 0.80:
            warnings.append(
                f"Answers are highly similar "
                f"({similarity:.2f})."
            )


        # ------------------------------------------------
        # 3. Problem quality
        # ------------------------------------------------

        problem = example["problem"].strip()

        if len(problem) < 20:
            errors.append(
                "Problem statement is too short."
            )


        # ------------------------------------------------
        # 4. Score validation
        # ------------------------------------------------

        for candidate, scores in {
            "A": scores_a,
            "B": scores_b
        }.items():

            for criterion in self.criteria:

                value = scores.get(criterion)

                if not isinstance(value, int):

                    errors.append(
                        f"{candidate}.{criterion} "
                        "is not an integer."
                    )

                elif not 0 <= value <= 10:

                    errors.append(
                        f"{candidate}.{criterion} "
                        "is outside 0-10 range."
                    )


        # ------------------------------------------------
        # 5. Feedback validation
        # ------------------------------------------------

        generic_feedback_patterns = [

            "answer is good",

            "answer is correct",

            "answer is incorrect",

            "better than the other answer",

            "this answer is better"
        ]


        for candidate, scores in {
            "A": scores_a,
            "B": scores_b
        }.items():

            feedback = (
                scores.get("feedback", "")
                .strip()
            )

            if len(feedback) < 15:

                errors.append(
                    f"{candidate} feedback is too short."
                )

            feedback_lower = feedback.lower()

            for pattern in generic_feedback_patterns:

                if (
                    feedback_lower == pattern
                    or feedback_lower == pattern + "."
                ):

                    warnings.append(
                        f"{candidate} feedback is too generic."
                    )


        # ------------------------------------------------
        # 6. Internal score consistency
        # ------------------------------------------------

        average_a = sum(
            scores_a[c]
            for c in self.criteria
        ) / len(self.criteria)

        average_b = sum(
            scores_b[c]
            for c in self.criteria
        ) / len(self.criteria)


        # Perfect scores should have meaningful feedback
        for candidate, scores in {
            "A": scores_a,
            "B": scores_b
        }.items():

            perfect_count = sum(
                1
                for criterion in self.criteria
                if scores[criterion] == 10
            )

            if perfect_count == 5:

                warnings.append(
                    f"{candidate} received all perfect scores. "
                    "Review for score inflation."
                )


        # ------------------------------------------------
        # 7. Score difference analysis
        # ------------------------------------------------

        score_difference = abs(
            average_a - average_b
        )

        if score_difference > 8:

            warnings.append(
                "Extremely large score difference."
            )

        # Both answers almost identical but scores far apart
        if (
            similarity > 0.80
            and score_difference > 3
        ):

            warnings.append(
                "Highly similar answers have "
                "very different scores."
            )


        # ------------------------------------------------
        # Final result
        # ------------------------------------------------

        return {
            "valid": len(errors) == 0,

            "errors": errors,

            "warnings": warnings,

            "metrics": {
                "answer_similarity": round(
                    similarity,
                    4
                ),

                "average_score_a": round(
                    average_a,
                    2
                ),

                "average_score_b": round(
                    average_b,
                    2
                ),

                "score_difference": round(
                    score_difference,
                    2
                )
            }
        }

In [ ]:
quality_validator = DatasetQualityValidator()

print("Quality validator initialized.")

In [ ]:
import json
import time
from datetime import datetime
from pydantic import ValidationError


class DatasetGenerationPipeline:

    def __init__(
        self,
        router,
        storage,
        checkpoint_manager,
        quality_validator,
        project_root=MODEL_DIR
    ):

        self.router = router
        self.storage = storage
        self.checkpoint_manager = checkpoint_manager
        self.quality_validator = quality_validator
        self.project_root = Path(project_root)

        self.validated_dir = (
            self.project_root
            / "data"
            / "validated"
        )

        self.validated_dir.mkdir(
            parents=True,
            exist_ok=True
        )


    def _clean_json_response(
        self,
        response_content
    ):

        cleaned = response_content.strip()

        # Remove markdown fences
        cleaned = re.sub(
            r"^```json\s*",
            "",
            cleaned,
            flags=re.IGNORECASE
        )

        cleaned = re.sub(
            r"^```\s*",
            "",
            cleaned
        )

        cleaned = re.sub(
            r"\s*```$",
            "",
            cleaned
        )

        return cleaned.strip()


    def _parse_and_validate(
        self,
        response_content
    ):

        try:

            cleaned = self._clean_json_response(
                response_content
            )

            data = json.loads(cleaned)

            example = TrainingExample.model_validate(
                data
            )

            return example, None

        except json.JSONDecodeError as error:

            return None, (
                f"JSON parsing error: {error}"
            )

        except ValidationError as error:

            return None, (
                f"Schema validation error: {error}"
            )

        except Exception as error:

            return None, (
                f"Unexpected validation error: {error}"
            )


    def _save_validated_example(
        self,
        example,
        task,
        response,
        quality_result
    ):

        filepath = (
            self.validated_dir
            / f"{example.id}.json"
        )

        data = example.model_dump()

        data["metadata"].update({

            "generator_provider":
                response.provider,

            "generator_model":
                response.model,

            "generation_latency_seconds":
                response.latency_seconds,

            "failure_type_target":
                task["failure_type"],

            "answer_relation_target":
                task["answer_relation"],

            "generated_at":
                datetime.now().isoformat(),

            "quality_metrics":
                quality_result["metrics"],

            "quality_warnings":
                quality_result["warnings"]
        })

        with open(
            filepath,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                data,
                file,
                indent=2,
                ensure_ascii=False
            )

        return filepath


    def process_task(
        self,
        task,
        provider="auto"
    ):

        # ------------------------------------------
        # 1. BUILD PROMPT
        # ------------------------------------------

        prompt = build_generation_prompt(

            category=task["category"],

            subcategory=task["subcategory"],

            difficulty=task["difficulty"],

            answer_relation=(
                task["answer_relation"]
            ),

            example_id=task["id"],

            failure_type=task["failure_type"]
        )


        # ------------------------------------------
        # 2. CALL API
        # ------------------------------------------

        response = self.router.generate(

            prompt=prompt,

            provider=provider
        )


        # ------------------------------------------
        # 3. HANDLE API FAILURE
        # ------------------------------------------

        if not response.success:

            self.storage.save_failure(

                prompt=prompt,

                response=response,

                metadata={
                    "task": task,
                    "stage": "api_generation"
                }
            )

            return {

                "success": False,

                "id": task["id"],

                "stage": "api_generation",

                "error": response.error
            }


        # ------------------------------------------
        # 4. SAVE RAW RESPONSE IMMEDIATELY
        # ------------------------------------------

        self.storage.save_success(

            prompt=prompt,

            response=response,

            metadata={
                "task": task,
                "stage": "raw_generation"
            }
        )


        # ------------------------------------------
        # 5. PARSE + PYDANTIC VALIDATION
        # ------------------------------------------

        example, validation_error = (
            self._parse_and_validate(
                response.content
            )
        )


        if validation_error:

            return {

                "success": False,

                "id": task["id"],

                "stage": "schema_validation",

                "error": validation_error
            }


        # ------------------------------------------
        # 6. LOCAL QUALITY VALIDATION
        # ------------------------------------------

        quality_result = (
            self.quality_validator.validate(
                example
            )
        )


        if not quality_result["valid"]:

            return {

                "success": False,

                "id": task["id"],

                "stage": "quality_validation",

                "error": quality_result["errors"],

                "warnings": quality_result[
                    "warnings"
                ]
            }


        # ------------------------------------------
        # 7. SAVE VALIDATED EXAMPLE
        # ------------------------------------------

        validated_path = (
            self._save_validated_example(

                example=example,

                task=task,

                response=response,

                quality_result=quality_result
            )
        )


        return {

            "success": True,

            "id": task["id"],

            "provider": response.provider,

            "model": response.model,

            "latency_seconds": (
                response.latency_seconds
            ),

            "warnings": (
                quality_result["warnings"]
            ),

            "path": str(validated_path)
        }


    def run(
        self,
        tasks,
        limit=None,
        provider="auto",
        delay_seconds=1
    ):

        checkpoint = (
            self.checkpoint_manager.load()
        )

        completed_ids = set(
            checkpoint.get(
                "completed_ids",
                []
            )
        )

        failed_ids = set(
            checkpoint.get(
                "failed_ids",
                []
            )
        )


        # Only process tasks not completed
        pending_tasks = [

            task

            for task in tasks

            if task["id"]
            not in completed_ids
        ]


        if limit is not None:

            pending_tasks = (
                pending_tasks[:limit]
            )


        print("\n" + "=" * 70)

        print("DATASET GENERATION PIPELINE")

        print("=" * 70)

        print(
            f"Total tasks: {len(tasks)}"
        )

        print(
            f"Completed previously: "
            f"{len(completed_ids)}"
        )

        print(
            f"Pending tasks: "
            f"{len(pending_tasks)}"
        )

        print("=" * 70)


        results = []


        for index, task in enumerate(
            pending_tasks,
            start=1
        ):

            print(
                f"\n[{index}/{len(pending_tasks)}] "
                f"{task['id']}"
            )

            print(
                f"Category: "
                f"{task['category']}"
            )


            result = self.process_task(

                task=task,

                provider=provider
            )


            results.append(result)


            # --------------------------------------
            # UPDATE CHECKPOINT
            # --------------------------------------

            if result["success"]:

                completed_ids.add(
                    task["id"]
                )

                print("✓ SUCCESS")

                print(
                    f"Provider: "
                    f"{result['provider']}"
                )

                print(
                    f"Latency: "
                    f"{result['latency_seconds']} sec"
                )

                if result["warnings"]:

                    print(
                        f"Warnings: "
                        f"{result['warnings']}"
                    )


            else:

                failed_ids.add(
                    task["id"]
                )

                print("✗ FAILED")

                print(
                    f"Stage: "
                    f"{result['stage']}"
                )

                print(
                    f"Error: "
                    f"{result['error']}"
                )


            # Save checkpoint after EVERY task
            self.checkpoint_manager.save(

                completed_ids=completed_ids,

                failed_ids=failed_ids
            )


            # Delay only between requests
            if index < len(pending_tasks):

                time.sleep(
                    delay_seconds
                )


        return results

In [ ]:
checkpoint_manager = CheckpointManager()

quality_validator = DatasetQualityValidator()

pipeline = DatasetGenerationPipeline(

    router=generator_router,

    storage=storage,

    checkpoint_manager=checkpoint_manager,

    quality_validator=quality_validator,

    project_root=MODEL_DIR
)

print("Final dataset generation pipeline initialized.")

In [ ]:
from collections import defaultdict
import time


class ProviderManager:

    def __init__(
        self,
        providers=None,
        max_retries_per_provider=2
    ):

        if providers is None:
            providers = [
                "gemini",
                "groq",
                "openrouter"
            ]

        self.providers = providers
        self.max_retries_per_provider = (
            max_retries_per_provider
        )

        self.current_index = 0

        self.stats = defaultdict(
            lambda: {
                "success": 0,
                "failure": 0,
                "total_latency": 0.0
            }
        )


    def get_next_provider(self):

        provider = self.providers[
            self.current_index
        ]

        self.current_index = (
            self.current_index + 1
        ) % len(self.providers)

        return provider


    def record_success(
        self,
        provider,
        latency
    ):

        self.stats[provider][
            "success"
        ] += 1

        self.stats[provider][
            "total_latency"
        ] += latency


    def record_failure(
        self,
        provider
    ):

        self.stats[provider][
            "failure"
        ] += 1


    def get_stats(self):

        results = {}

        for provider, data in self.stats.items():

            total_requests = (
                data["success"]
                + data["failure"]
            )

            average_latency = 0

            if data["success"] > 0:

                average_latency = (
                    data["total_latency"]
                    / data["success"]
                )

            results[provider] = {

                "total_requests":
                    total_requests,

                "success":
                    data["success"],

                "failure":
                    data["failure"],

                "average_latency":
                    round(
                        average_latency,
                        2
                    )
            }

        return results

In [ ]:
generator_router = GeneratorRouter()

print("\nREGISTERED MODELS:")
for provider, generator in generator_router.generators.items():
    print(
        f"{provider}: {generator.model_name}"
    )

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import json


PROVIDER_STATE_PATH = (
    MODEL_DIR
    / "checkpoints"
    / "provider_state.json"
)


class PersistentProviderManager:

    def __init__(
        self,
        providers=None,
        state_path=PROVIDER_STATE_PATH
    ):

        self.providers = providers or [
            "gemini",
            "groq",
            "openrouter"
        ]

        self.state_path = Path(state_path)

        self.state = self._load_state()


    def _default_provider_state(self):

        return {
            "success": 0,
            "failure": 0,
            "total_latency": 0.0,
            "cooldown_until": None,
            "last_error": None,
            "last_used": None
        }


    def _default_state(self):

        return {
            "current_index": 0,
            "providers": {
                provider: self._default_provider_state()
                for provider in self.providers
            },
            "updated_at": None
        }


    def _load_state(self):

        if not self.state_path.exists():

            return self._default_state()

        try:

            with open(
                self.state_path,
                "r",
                encoding="utf-8"
            ) as file:

                state = json.load(file)

            # Add missing providers safely
            for provider in self.providers:

                if provider not in state["providers"]:

                    state["providers"][provider] = (
                        self._default_provider_state()
                    )

            return state

        except Exception:

            print(
                "Warning: Could not load provider state. "
                "Creating a new state."
            )

            return self._default_state()


    def save_state(self):

        self.state["updated_at"] = (
            datetime.now().isoformat()
        )

        self.state_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with open(
            self.state_path,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                self.state,
                file,
                indent=2
            )


    def is_available(self, provider):

        provider_state = (
            self.state["providers"][provider]
        )

        cooldown_until = (
            provider_state.get("cooldown_until")
        )

        if cooldown_until is None:

            return True

        cooldown_time = datetime.fromisoformat(
            cooldown_until
        )

        # Cooldown expired
        if datetime.now() >= cooldown_time:

            provider_state["cooldown_until"] = None

            self.save_state()

            return True

        return False


    def get_next_provider(self):

        total = len(self.providers)

        for _ in range(total):

            index = self.state["current_index"]

            provider = self.providers[index]

            self.state["current_index"] = (
                index + 1
            ) % total

            self.save_state()

            if self.is_available(provider):

                return provider

        return None


    def record_success(
        self,
        provider,
        latency
    ):

        provider_state = (
            self.state["providers"][provider]
        )

        provider_state["success"] += 1

        provider_state["total_latency"] += latency

        provider_state["last_used"] = (
            datetime.now().isoformat()
        )

        provider_state["last_error"] = None

        self.save_state()


    def record_failure(
        self,
        provider,
        error=None
    ):

        provider_state = (
            self.state["providers"][provider]
        )

        provider_state["failure"] += 1

        provider_state["last_error"] = str(error)

        provider_state["last_used"] = (
            datetime.now().isoformat()
        )

        self.save_state()


    def set_cooldown(
        self,
        provider,
        seconds,
        reason=None
    ):

        cooldown_until = (
            datetime.now()
            + timedelta(seconds=seconds)
        )

        provider_state = (
            self.state["providers"][provider]
        )

        provider_state["cooldown_until"] = (
            cooldown_until.isoformat()
        )

        provider_state["last_error"] = reason

        self.save_state()

        print(
            f"{provider} placed on cooldown "
            f"for {seconds} seconds."
        )


    def get_stats(self):

        results = {}

        for provider, data in (
            self.state["providers"].items()
        ):

            success = data["success"]

            average_latency = 0.0

            if success > 0:

                average_latency = round(
                    data["total_latency"] / success,
                    2
                )

            results[provider] = {

                "success": success,

                "failure": data["failure"],

                "average_latency": average_latency,

                "cooldown_until":
                    data["cooldown_until"],

                "last_error":
                    data["last_error"]
            }

        return results

In [ ]:
COOLDOWN_CONFIG = {

    "rate_limit": 60,

    "temporary_failure": 30,

    "provider_error": 20
}

In [ ]:
class PersistentSmartRouter:

    def __init__(
        self,
        base_router,
        provider_manager,
        cooldown_config=COOLDOWN_CONFIG
    ):

        self.base_router = base_router

        self.provider_manager = (
            provider_manager
        )

        self.cooldown_config = (
            cooldown_config
        )


    def generate(
        self,
        prompt,
        provider="auto"
    ):

        # ------------------------------------------
        # Manual provider mode
        # ------------------------------------------

        if provider != "auto":

            response = (
                self.base_router.generate(
                    prompt=prompt,
                    provider=provider
                )
            )

            self._handle_response(
                provider,
                response
            )

            return response


        # ------------------------------------------
        # Automatic provider rotation
        # ------------------------------------------

        attempted_providers = []

        total_providers = len(
            self.provider_manager.providers
        )


        for _ in range(total_providers):

            selected_provider = (
                self.provider_manager
                .get_next_provider()
            )


            # All providers on cooldown
            if selected_provider is None:

                break


            attempted_providers.append(
                selected_provider
            )


            print(
                f"Trying provider: "
                f"{selected_provider}"
            )


            response = (
                self.base_router.generate(

                    prompt=prompt,

                    provider=selected_provider
                )
            )


            self._handle_response(

                selected_provider,

                response
            )


            if response.success:

                return response


        # ------------------------------------------
        # All providers failed
        # ------------------------------------------

        if attempted_providers:

            response.error = (

                "All available providers failed. "
                f"Attempted: {attempted_providers}. "
                f"Last error: {response.error}"
            )

            return response


        # No providers available
        return type(
            "UnavailableResponse",
            (),
            {
                "success": False,

                "content": None,

                "provider": None,

                "model": None,

                "latency_seconds": 0,

                "error": (
                    "All providers are currently "
                    "on cooldown."
                )
            }
        )()


    def _handle_response(
        self,
        provider,
        response
    ):

        if response.success:

            self.provider_manager.record_success(

                provider,

                response.latency_seconds
            )

            return


        # Record failure
        self.provider_manager.record_failure(

            provider,

            response.error
        )


        # ------------------------------------------
        # Rate limit
        # ------------------------------------------

        if detect_rate_limit(response.error):

            self.provider_manager.set_cooldown(

                provider,

                seconds=(
                    self.cooldown_config[
                        "rate_limit"
                    ]
                ),

                reason=response.error
            )

            return


        # ------------------------------------------
        # Other temporary failure
        # ------------------------------------------

        self.provider_manager.set_cooldown(

            provider,

            seconds=(
                self.cooldown_config[
                    "temporary_failure"
                ]
            ),

            reason=response.error
        )

In [ ]:
persistent_provider_manager = (
    PersistentProviderManager(

        providers=[
            "gemini",
            "groq",
            "openrouter"
        ]
    )
)


persistent_router = PersistentSmartRouter(

    base_router=generator_router,

    provider_manager=(
        persistent_provider_manager
    )
)


print(
    "Persistent provider router initialized."
)

In [ ]:
def detect_rate_limit(error_message):

    if not error_message:

        return False

    error_text = str(
        error_message
    ).lower()

    rate_limit_patterns = [

        "rate limit",

        "rate_limit",

        "too many requests",

        "429",

        "quota exceeded",

        "resource exhausted",

        "requests per minute",

        "tokens per minute",

        "rpm limit",

        "tpm limit"
    ]

    return any(
        pattern in error_text
        for pattern in rate_limit_patterns
    )

In [ ]:
pipeline = DatasetGenerationPipeline(

    router=persistent_router,

    storage=storage,

    checkpoint_manager=checkpoint_manager,

    quality_validator=quality_validator,

    project_root=MODEL_DIR
)

print(
    "Pipeline connected to persistent router."
)

In [ ]:
validated_files = list(
    (MODEL_DIR / "data" / "validated").glob("*.json")
)

print(
    f"Validated examples: {len(validated_files)}"
)

for file in validated_files[:5]:
    print(file.name)

In [ ]:
import inspect

print(inspect.getsource(GeneratorRouter.__init__))

In [ ]:
persistent_router = PersistentSmartRouter(
    base_router=generator_router,
    provider_manager=persistent_provider_manager
)

pipeline = DatasetGenerationPipeline(
    router=persistent_router,
    storage=storage,
    checkpoint_manager=checkpoint_manager,
    quality_validator=quality_validator,
    project_root=MODEL_DIR
)

print("Pipeline ready.")

In [ ]:
persistent_provider_manager.state["providers"]["openrouter"]["cooldown_until"] = None
persistent_provider_manager.state["providers"]["openrouter"]["last_error"] = None

persistent_provider_manager.save_state()

In [ ]:
print("Project root:", MODEL_DIR)
print("Storage:", storage)
print("Checkpoint manager:", checkpoint_manager)

In [ ]:
results_batch_16 = pipeline.run(
    tasks=tasks,
    limit=250,
    provider="auto",
    delay_seconds=2.4
)

In [ ]:
successful = [
    r for r in results_batch_16
    if r["success"]
]

failed = [
    r for r in results_batch_16
    if not r["success"]
]

print("=" * 70)
print("BATCH 16 SUMMARY")
print("=" * 70)

print("Attempted:", len(results_batch_16))
print("Successful:", len(successful))
print("Failed:", len(failed))

print("\nProvider Statistics:")
print(json.dumps(
    persistent_provider_manager.get_stats(),
    indent=2
))

#Dataset downloading

In [ ]:
DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/chatbot_arena_llm_judges"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
from datasets import load_dataset

ds = load_dataset("potsawee/chatbot-arena-llm-judges")

ds.save_to_disk(DATA_DIR)

print(f"Dataset saved to: {DATA_DIR}")

AttributeError: partially initialized module 'datasets' from '/usr/local/lib/python3.13/dist-packages/datasets/__init__.py' has no attribute 'utils' (most likely due to a circular import)

In [ ]:
from datasets import load_from_disk

ds = load_from_disk(
    "/content/drive/MyDrive/LLM-Judge/data/public/chatbot_arena_llm_judges"
)

print(ds)

In [ ]:
from datasets import load_from_disk

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/chatbot_arena_llm_judges"

ds = load_from_disk(DATA_DIR)

print(ds)
print(ds["train"].column_names)
print(ds["train"].features)

In [ ]:
print(ds["train"][0])

In [ ]:
from datasets import load_from_disk, Dataset
import os
import json


SOURCE_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/chatbot_arena_llm_judges"

OUTPUT_DIR = "/content/drive/MyDrive/LLM-Judge/data/processed/chatbot_arena"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_winner(row):
    if row["winner_model_a"] == 1:
        return "A"

    if row["winner_model_b"] == 1:
        return "B"

    if row["winner_tie"] == 1:
        return "tie"

    return None


def convert_dataset():

    ds = load_from_disk(SOURCE_DIR)

    examples = []

    for row in ds["train"]:

        winner = get_winner(row)

        if winner is None:
            continue

        example = {
            "question": row["question"],
            "answer_a": row["answer_a"],
            "answer_b": row["answer_b"],
            "winner": winner
        }

        examples.append(example)

    converted = Dataset.from_list(examples)

    output_path = os.path.join(OUTPUT_DIR, "train")

    converted.save_to_disk(output_path)

    print(f"Converted examples: {len(converted)}")
    print(f"Saved to: {output_path}")


if __name__ == "__main__":
    convert_dataset()

In [ ]:
!python /content/drive/MyDrive/LLM-Judge/src/training/convert_chatbot_arena.py

In [ ]:
from datasets import load_from_disk

path = "/content/drive/MyDrive/LLM-Judge/data/processed/chatbot_arena/train"

arena = load_from_disk(path)

print(arena)
print(arena[0])

In [ ]:
from datasets import load_dataset
import os

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/900k_judgements"

os.makedirs(DATA_DIR, exist_ok=True)

ds = load_dataset("ibm-research/900K-Judgements")

ds.save_to_disk(DATA_DIR)

print(f"Dataset saved to: {DATA_DIR}")

In [ ]:
from datasets import load_dataset
import os

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/code-edit-dpo"

os.makedirs(DATA_DIR, exist_ok=True)
ds = load_dataset("PotatoHD/code-edit-dpo")
ds.save_to_disk(DATA_DIR)

print(f"Dataset saved to: {DATA_DIR}")

In [ ]:
from datasets import load_dataset
import os

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/web_bench"
os.makedirs(DATA_DIR, exist_ok=True)

ds = load_dataset("bytedance-research/Web-Bench")
ds.save_to_disk(DATA_DIR)

print(f"Saved to: {DATA_DIR}")
print(ds)

In [ ]:
from datasets import load_dataset
import os

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/swebench_verified"
os.makedirs(DATA_DIR, exist_ok=True)

ds = load_dataset("SWE-bench/SWE-bench_Verified")
ds.save_to_disk(DATA_DIR)

print(f"Saved to: {DATA_DIR}")
print(ds)

In [ ]:
from datasets import load_dataset
import os

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/livecodebench"

os.makedirs(DATA_DIR, exist_ok=True)

data_files = {
    "test": [
        "https://huggingface.co/datasets/livecodebench/code_generation_lite/resolve/main/test5.jsonl",
        "https://huggingface.co/datasets/livecodebench/code_generation_lite/resolve/main/test6.jsonl"
    ]
}

lcb_codegen = load_dataset(
    "json",
    data_files=data_files
)

print(lcb_codegen)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "livecodebench/code_generation_lite",
    repo_type="dataset"
)

for f in files:
    print(f)

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

In [ ]:
from datasets import load_from_disk, Dataset
import os
import json


SOURCE_DIR = (
    "/content/drive/MyDrive/LLM-Judge/"
    "data/processed/chatbot_arena/train"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/LLM-Judge/"
    "data/training/arena_sft"
)


SYSTEM_PROMPT = """You are an impartial AI response judge.

Given a question and two candidate answers, determine which answer is better.

Evaluate the answers based on:
- correctness
- relevance
- completeness
- reasoning quality
- clarity

Do not consider the identity or reputation of the model that produced an answer.

Return only valid JSON in this format:

{"winner":"A"}

where winner must be A, B, or tie.
"""


def clean_text(value):
    """Safely convert a value to a non-empty string."""
    if value is None:
        return None

    if not isinstance(value, str):
        value = str(value)

    value = value.strip()

    if not value:
        return None

    return value


def create_prompt(question, answer_a, answer_b):

    return f"""Question:
{question}

Answer A:
{answer_a}

Answer B:
{answer_b}"""


def create_example(question, answer_a, answer_b, winner):

    prompt = create_prompt(
        question,
        answer_a,
        answer_b
    )

    assistant_response = json.dumps(
        {"winner": winner},
        separators=(",", ":")
    )

    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{assistant_response}"
        "<|im_end|>"
    )

    return {"text": text}


def main():

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    dataset = load_from_disk(SOURCE_DIR)

    print("Loaded dataset:")
    print(dataset)

    examples = []
    skipped = 0

    for row in dataset:

        question = clean_text(row["question"])
        answer_a = clean_text(row["answer_a"])
        answer_b = clean_text(row["answer_b"])
        winner = row["winner"]

        # Skip incomplete examples
        if (
            question is None
            or answer_a is None
            or answer_b is None
            or winner not in ["A", "B", "tie"]
        ):
            skipped += 1
            continue

        examples.append(
            create_example(
                question,
                answer_a,
                answer_b,
                winner
            )
        )

    result = Dataset.from_list(examples)

    print("\nProcessing complete.")
    print("Valid examples:", len(result))
    print("Skipped examples:", skipped)

    output_path = os.path.join(
        OUTPUT_DIR,
        "train"
    )

    # Remove old output if it exists
    if os.path.exists(output_path):
        import shutil
        shutil.rmtree(output_path)

    result.save_to_disk(output_path)

    print("\nSaved to:")
    print(output_path)

    print("\nSample:")
    print(result[0]["text"])


if __name__ == "__main__":
    main()

In [ ]:
!python /content/drive/MyDrive/LLM-Judge/src/training/create_arena_sft.py

In [ ]:
from datasets import load_from_disk

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/900k_judgements"

ds_900k = load_from_disk(DATA_DIR)

print(ds_900k)

for split in ds_900k:
    print("\nSplit:", split)
    print("Rows:", len(ds_900k[split]))
    print("Columns:", ds_900k[split].column_names)

print("\nFirst example:")
print(ds_900k[list(ds_900k.keys())[0]][0])

In [ ]:
from datasets import load_from_disk

DATA_DIR = "/content/drive/MyDrive/LLM-Judge/data/public/900k_judgements"

ds_900k = load_from_disk(DATA_DIR)

print(ds_900k)

print("\nColumns:")
print(ds_900k["train"].column_names)

print("\nNumber of rows:")
print(len(ds_900k["train"]))

print("\nFirst example:")
print(ds_900k["train"][0])

In [ ]:
for i in range(5):
    print("=" * 80)
    print(ds_900k["train"][i])

In [ ]:
import os

BASE = "/content/drive/MyDrive/LLM-Judge/data/public"

print("Folders in data/public:\n")

for name in sorted(os.listdir(BASE)):
    path = os.path.join(BASE, name)

    if os.path.isdir(path):
        print(name)

In [ ]:
from datasets import load_from_disk, Dataset, concatenate_datasets
import os
import random


ARENA_PATH = (
    "/content/drive/MyDrive/LLM-Judge/"
    "data/processed/chatbot_arena/train"
)

CODE_DPO_PATH = (
    "/content/drive/MyDrive/LLM-Judge/"
    "data/public/code-edit-dpo"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/LLM-Judge/"
    "data/training/unified"
)

CODE_DPO_SAMPLES = 10000

SEED = 42


def prepare_arena():

    dataset = load_from_disk(ARENA_PATH)

    examples = []

    for row in dataset:

        examples.append({
            "question": row["question"],
            "answer_a": row["answer_a"],
            "answer_b": row["answer_b"],
            "winner": row["winner"],
            "source": "chatbot_arena"
        })

    return Dataset.from_list(examples)


def prepare_code_dpo():

    dataset_dict = load_from_disk(CODE_DPO_PATH)

    # Get the actual training split
    dataset = dataset_dict["train"]

    # Shuffle before sampling
    dataset = dataset.shuffle(seed=SEED)

    # Select up to CODE_DPO_SAMPLES examples
    dataset = dataset.select(
        range(min(CODE_DPO_SAMPLES, len(dataset)))
    )

    examples = []

    rng = random.Random(SEED)

    for row in dataset:

        prompt = row["prompt"]
        chosen = row["chosen"]
        rejected = row["rejected"]

        if not prompt or not chosen or not rejected:
            continue

        # Randomize A/B position
        if rng.random() < 0.5:
            answer_a = chosen
            answer_b = rejected
            winner = "A"
        else:
            answer_a = rejected
            answer_b = chosen
            winner = "B"

        examples.append({
            "question": prompt,
            "answer_a": answer_a,
            "answer_b": answer_b,
            "winner": winner,
            "source": "code_edit_dpo"
        })

    return Dataset.from_list(examples)


def main():

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Preparing Chatbot Arena...")
    arena = prepare_arena()

    print(f"Arena examples: {len(arena)}")

    print("\nPreparing Code Edit DPO...")
    code_dpo = prepare_code_dpo()

    print(f"Code DPO examples: {len(code_dpo)}")

    print("\nCombining datasets...")

    unified = concatenate_datasets([
        arena,
        code_dpo
    ])

    # Shuffle final dataset
    unified = unified.shuffle(seed=SEED)

    print("\nUnified dataset:")
    print(unified)

    # 90% train / 10% validation
    split = unified.train_test_split(
        test_size=0.10,
        seed=SEED
    )

    train_dataset = split["train"]
    validation_dataset = split["test"]

    train_path = os.path.join(
        OUTPUT_DIR,
        "train"
    )

    validation_path = os.path.join(
        OUTPUT_DIR,
        "validation"
    )

    # Remove old datasets if present
    import shutil

    if os.path.exists(train_path):
        shutil.rmtree(train_path)

    if os.path.exists(validation_path):
        shutil.rmtree(validation_path)

    train_dataset.save_to_disk(train_path)
    validation_dataset.save_to_disk(validation_path)

    print("\n========================================")
    print("DATASET CREATED")
    print("========================================")

    print(f"Total:      {len(unified)}")
    print(f"Train:      {len(train_dataset)}")
    print(f"Validation: {len(validation_dataset)}")

    print("\nTrain:")
    print(train_path)

    print("\nValidation:")
    print(validation_path)

    print("\nExample:")
    print(train_dataset[0])


if __name__ == "__main__":
    main()

In [ ]:
!python /content/drive/MyDrive/LLM-Judge/src/training/build_training_dataset.py

In [ ]:
from datasets import load_from_disk
import os


BASE_DIR = "/content/drive/MyDrive/LLM-Judge/data/training/unified"

OUTPUT_DIR = "/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted"


SYSTEM_PROMPT = """You are an impartial AI response judge.

Given a question and two candidate answers, determine which answer is better.

Evaluate the answers based on:
- correctness
- relevance
- completeness
- reasoning quality
- clarity

Do not consider the identity or reputation of the model that produced an answer.

Return only JSON in this format:

{"winner":"A"}

The winner must be exactly one of:
A
B
tie
"""


def format_dataset(dataset):

    formatted = []

    for row in dataset:

        question = row["question"]
        answer_a = row["answer_a"]
        answer_b = row["answer_b"]
        winner = row["winner"]

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Answer A:\n{answer_a}\n\n"
                    f"Answer B:\n{answer_b}"
                )
            },
            {
                "role": "assistant",
                "content": f'{{"winner":"{winner}"}}'
            }
        ]

        formatted.append({
            "messages": messages,
            "source": row["source"]
        })

    return formatted


def save_split(split_name):

    input_path = os.path.join(BASE_DIR, split_name)

    dataset = load_from_disk(input_path)

    print(f"\nLoaded {split_name}: {len(dataset)} examples")

    formatted = format_dataset(dataset)

    output_path = os.path.join(
        OUTPUT_DIR,
        split_name
    )

    os.makedirs(output_path, exist_ok=True)

    from datasets import Dataset

    formatted_dataset = Dataset.from_list(formatted)

    formatted_dataset.save_to_disk(output_path)

    print(f"Saved {split_name}: {len(formatted_dataset)}")
    print(f"Path: {output_path}")


def main():

    save_split("train")
    save_split("validation")

    print("\nFormatting complete.")


if __name__ == "__main__":
    main()

In [ ]:
!python /content/drive/MyDrive/LLM-Judge/src/training/format_qwen_dataset.py

In [ ]:
from datasets import load_from_disk

path = "/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted/train"

ds = load_from_disk(path)

print(ds)
print(ds[0]["messages"])

In [ ]:
from datasets import load_from_disk

TRAIN_PATH = "/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted/train"
VAL_PATH = "/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted/validation"

train_ds = load_from_disk(TRAIN_PATH)
val_ds = load_from_disk(VAL_PATH)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))

In [ ]:
import os

# Set before importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ============================================================
# PATHS
# ============================================================

MODEL_PATH = (
    "/content/drive/MyDrive/LLM-Judge/"
    "models/Qwen2.5-7B-Instruct"
)

TRAIN_PATH = (
    "/content/drive/MyDrive/LLM-Judge/"
    "data/training/qwen_formatted/train"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/LLM-Judge/"
    "outputs/qwen2.5-7b-judge-qlora-v1"
)


# ============================================================
# TRAINING SETTINGS
# ============================================================

MAX_LENGTH = 1536

TRAIN_BATCH_SIZE = 1

GRADIENT_ACCUMULATION_STEPS = 16

LEARNING_RATE = 1e-4

NUM_EPOCHS = 1

WARMUP_STEPS = 300


# Token budgets inside the prompt.
# This keeps the final assistant answer inside MAX_LENGTH.
QUESTION_MAX_TOKENS = 384
ANSWER_MAX_TOKENS = 480


# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an impartial AI response judge.

Given a question and two candidate answers, determine which answer is better.

Evaluate the answers based on:
- correctness
- relevance
- completeness
- reasoning quality
- clarity

Do not consider the identity or reputation of the model that produced an answer.

Return only JSON in this format:

{"winner":"A"}

The winner must be exactly one of:
A
B
tie
"""


# ============================================================
# DATA COLLATOR
# ============================================================

class JudgeDataCollator:

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):

        input_ids = [
            feature["input_ids"]
            for feature in features
        ]

        attention_masks = [
            feature["attention_mask"]
            for feature in features
        ]

        labels = [
            feature["labels"]
            for feature in features
        ]

        batch = self.tokenizer.pad(
            {
                "input_ids": input_ids,
                "attention_mask": attention_masks,
            },
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]

        padded_labels = []

        for label in labels:

            padding_length = max_len - len(label)

            padded_labels.append(
                label + [-100] * padding_length
            )

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch


# ============================================================
# TEXT TRUNCATION
# ============================================================

def truncate_text(
    text,
    tokenizer,
    max_tokens,
):
    """
    Truncate text by tokens while preserving valid text.
    """

    if text is None:
        return ""

    text = str(text).strip()

    if not text:
        return ""

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_tokens,
    )["input_ids"]

    return tokenizer.decode(
        token_ids,
        skip_special_tokens=True,
    )


# ============================================================
# TOKENIZATION
# ============================================================

def tokenize_example(example, tokenizer):

    question = truncate_text(
        example["messages"][1]["content"].split(
            "Answer A:"
        )[0].replace(
            "Question:",
            ""
        ).strip(),
        tokenizer,
        QUESTION_MAX_TOKENS,
    )

    original_user_content = example["messages"][1]["content"]

    # Extract original answers safely
    answer_a_marker = "Answer A:"
    answer_b_marker = "Answer B:"

    if (
        answer_a_marker in original_user_content
        and answer_b_marker in original_user_content
    ):

        answer_section = original_user_content.split(
            answer_a_marker,
            1
        )[1]

        answer_a, answer_b = answer_section.split(
            answer_b_marker,
            1
        )

        answer_a = answer_a.strip()
        answer_b = answer_b.strip()

    else:
        raise ValueError(
            "Unexpected training example format."
        )

    answer_a = truncate_text(
        answer_a,
        tokenizer,
        ANSWER_MAX_TOKENS,
    )

    answer_b = truncate_text(
        answer_b,
        tokenizer,
        ANSWER_MAX_TOKENS,
    )

    winner = example["messages"][2]["content"]

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Answer A:\n{answer_a}\n\n"
                f"Answer B:\n{answer_b}"
            ),
        },
        {
            "role": "assistant",
            "content": winner,
        },
    ]

    # Prompt without assistant response
    prompt_messages = messages[:2]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full["input_ids"]

    attention_mask = full["attention_mask"]

    prompt_length = min(
        len(prompt_ids),
        len(input_ids),
    )

    labels = input_ids.copy()

    # Mask system + user tokens.
    for i in range(prompt_length):
        labels[i] = -100

    # Safety check:
    # ensure at least one supervised token exists.
    if all(label == -100 for label in labels):
        raise ValueError(
            "Example contains no trainable assistant tokens."
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


# ============================================================
# MAIN
# ============================================================

def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True,
    )

    # --------------------------------------------------------
    # GPU CHECK
    # --------------------------------------------------------

    print("=" * 60)
    print("GPU CHECK")
    print("=" * 60)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. "
            "Make sure Colab is using a T4 GPU."
        )

    gpu_name = torch.cuda.get_device_name(0)

    total_vram = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print("GPU:", gpu_name)
    print(
        "VRAM:",
        round(total_vram, 2),
        "GB",
    )

    torch.cuda.empty_cache()

    # --------------------------------------------------------
    # TOKENIZER
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("LOADING TOKENIZER")
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    print("Tokenizer loaded.")

    # --------------------------------------------------------
    # DATASETS
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("LOADING DATASETS")
    print("=" * 60)

    train_dataset = load_from_disk(
        TRAIN_PATH
    )

    print(
        "Train examples:",
        len(train_dataset),
    )

    # --------------------------------------------------------
    # TOKENIZATION
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("TOKENIZING TRAIN DATA")
    print("=" * 60)

    train_tokenized = train_dataset.map(
        lambda x: tokenize_example(
            x,
            tokenizer,
        ),
        remove_columns=train_dataset.column_names,
        desc="Tokenizing train",
    )

    print(
        "\nTokenized examples:",
        len(train_tokenized),
    )

    # --------------------------------------------------------
    # 4-BIT CONFIGURATION
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("CONFIGURING 4-BIT QUANTIZATION")
    print("=" * 60)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    print("4-bit configuration ready.")

    # --------------------------------------------------------
    # LOAD QWEN
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("LOADING QWEN 2.5 7B")
    print("=" * 60)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    model.config.use_cache = False

    print("Model loaded.")

    print(
        "GPU memory after model load:",
        round(
            torch.cuda.memory_allocated() / (1024 ** 3),
            2,
        ),
        "GB",
    )

    # --------------------------------------------------------
    # LoRA
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("CONFIGURING LoRA")
    print("=" * 60)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    # IMPORTANT:
    # Do NOT call prepare_model_for_kbit_training().
    # It caused a large FP32 memory spike on the T4.

    model = get_peft_model(
        model,
        lora_config,
    )

    # Required for gradient checkpointing
    # when the input embeddings are frozen.
    model.enable_input_require_grads()

    print("LoRA configuration ready.")

    model.print_trainable_parameters()

    # --------------------------------------------------------
    # TRAINING ARGUMENTS
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("CONFIGURING TRAINING")
    print("=" * 60)

    training_args = TrainingArguments(

        output_dir=OUTPUT_DIR,

        num_train_epochs=NUM_EPOCHS,

        per_device_train_batch_size=TRAIN_BATCH_SIZE,

        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        learning_rate=LEARNING_RATE,

        weight_decay=0.01,

        fp16=True,

        gradient_checkpointing=True,

        optim="paged_adamw_8bit",

        lr_scheduler_type="cosine",

        warmup_steps=WARMUP_STEPS,

        logging_steps=10,

        save_strategy="steps",

        save_steps=500,

        save_total_limit=2,

        report_to="none",

        remove_unused_columns=False,

        max_grad_norm=0.3,
    )

    print("Training configuration ready.")

    # --------------------------------------------------------
    # TRAINER
    # --------------------------------------------------------

    data_collator = JudgeDataCollator(
        tokenizer
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        data_collator=data_collator,
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("STARTING QLORA TRAINING")
    print("=" * 60)

    print(
        "Train examples:",
        len(train_tokenized),
    )

    print(
        "Max sequence length:",
        MAX_LENGTH,
    )

    print(
        "Batch size:",
        TRAIN_BATCH_SIZE,
    )

    print(
        "Gradient accumulation:",
        GRADIENT_ACCUMULATION_STEPS,
    )

    trainer.train()

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("SAVING LoRA ADAPTER")
    print("=" * 60)

    trainer.save_model(
        OUTPUT_DIR
    )

    tokenizer.save_pretrained(
        OUTPUT_DIR
    )

    print("\nTraining complete.")

    print(
        "LoRA adapter saved to:"
    )

    print(OUTPUT_DIR)


if __name__ == "__main__":
    main()

In [ ]:
!zip -r -q \
"/content/Qwen2.5-7B-Instruct.zip" \
"/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct"

In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct"

for item in sorted(os.listdir(MODEL_DIR)):
    path = os.path.join(MODEL_DIR, item)

    if os.path.isdir(path):
        print("[DIR] ", item)
    else:
        print("[FILE]", item)

In [ ]:
can I directly

# kaggle connection

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
from google.auth import default
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

creds, _ = default(scopes=SCOPES)

service = build(
    "drive",
    "v3",
    credentials=creds
)

print("Google Drive API connected successfully.")

Google Drive API connected successfully.


In [ ]:
FOLDER_ID = "1m5LV1x8Yf3eQ_9w8EW2rZppMcxD95gJH"

results = service.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed = false",
    fields="files(id,name,size,mimeType)",
    pageSize=100,
).execute()

files = results.get("files", [])

for f in sorted(files, key=lambda x: x["name"]):
    print(
        f"{f['name']}\n"
        f"ID: {f['id']}\n"
        f"Size: {f.get('size', 'N/A')}\n"
        f"Type: {f['mimeType']}\n"
        f"{'-' * 60}"
    )

.cache
ID: 10P-QGofO81ofGPstK6hcZoKdlMUfGYoJ
Size: N/A
Type: application/vnd.google-apps.folder
------------------------------------------------------------
.gitattributes
ID: 1k_cwzHB5eQttLln5qVXJlshwU7GRd1Iz
Size: 1519
Type: application/octet-stream
------------------------------------------------------------
LICENSE
ID: 1lgbmoFvSno3_21neXoTB_MfMoSgTWhHo
Size: 11343
Type: application/octet-stream
------------------------------------------------------------
README.md
ID: 1D8u-ONJhMA9Bmp4ZvimTECtiAEDopmj5
Size: 6240
Type: text/markdown
------------------------------------------------------------
config.json
ID: 1U-mFDEJ7Djaqv-jzIdzkCkb1XhyNrH-8
Size: 663
Type: application/json
------------------------------------------------------------
generation_config.json
ID: 1swpx5Pm8WXYgYzY5QXuD2Why6Sncm7Rh
Size: 243
Type: application/json
------------------------------------------------------------
merges.txt
ID: 1GMt_DJ0YHZbrE_cFXdBoTRuQPiM_kgcz
Size: 1671839
Type: text/plain
------------------

In [ ]:
import os

TRAIN_DIR = "/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted/train"

print("Dataset size:")
!du -sh "$TRAIN_DIR"

print("\nFiles:")
for f in sorted(os.listdir(TRAIN_DIR)):
    path = os.path.join(TRAIN_DIR, f)
    if os.path.isfile(path):
        print(f, f"{os.path.getsize(path) / (1024**2):.2f} MB")

Dataset size:
1.7G	/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted/train

Files:
cache-3ea4c2f6881f03ff.arrow 528.13 MB
cache-b7aeb6957386d36d.arrow 496.21 MB
cache-e57d4389b322bdc5.arrow 446.71 MB
data-00000-of-00001.arrow 222.29 MB
dataset_info.json 0.00 MB
state.json 0.00 MB


In [ ]:
import os
import shutil

SOURCE = "/content/drive/MyDrive/LLM-Judge/data/training/qwen_formatted/train"
CLEAN = "/content/qwen_train_clean"

if os.path.exists(CLEAN):
    shutil.rmtree(CLEAN)

os.makedirs(CLEAN, exist_ok=True)

KEEP = [
    "data-00000-of-00001.arrow",
    "dataset_info.json",
    "state.json",
]

for filename in KEEP:
    src = os.path.join(SOURCE, filename)
    dst = os.path.join(CLEAN, filename)

    if not os.path.exists(src):
        raise FileNotFoundError(src)

    shutil.copy2(src, dst)

print("Clean training dataset created.")

Clean training dataset created.


In [ ]:
!du -sh /content/qwen_train_clean

223M	/content/qwen_train_clean


In [ ]:
FOLDER_ID = "1m5LV1x8Yf3eQ_9w8EW2rZppMcxD95gJH"

In [ ]:
from googleapiclient.discovery import build
from google.auth import default

creds, _ = default(
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)

service = build(
    "drive",
    "v3",
    credentials=creds
)


def find_folder(name, parent_id):
    results = service.files().list(
        q=(
            f"'{parent_id}' in parents "
            f"and name = '{name}' "
            f"and mimeType = 'application/vnd.google-apps.folder' "
            f"and trashed = false"
        ),
        fields="files(id,name)",
        pageSize=20,
    ).execute()

    files = results.get("files", [])

    if not files:
        raise FileNotFoundError(
            f"Folder not found: {name}"
        )

    return files[0]["id"]


# Navigate through your existing Drive structure
root_id = "root"

llm_judge_id = find_folder(
    "LLM-Judge",
    root_id
)

data_id = find_folder(
    "data",
    llm_judge_id
)

training_id = find_folder(
    "training",
    data_id
)

qwen_formatted_id = find_folder(
    "qwen_formatted",
    training_id
)

train_id = find_folder(
    "train",
    qwen_formatted_id
)

print("Training folder ID:")
print(train_id)

Training folder ID:
1HRPAKiPRJGrM19UdpCtbwL0rOJYJozPU


In [ ]:
results = service.files().list(
    q=f"'{train_id}' in parents and trashed = false",
    fields="files(id,name,size,mimeType)",
    pageSize=20,
).execute()

train_files = results.get("files", [])

for f in sorted(train_files, key=lambda x: x["name"]):
    print(
        f"\nName: {f['name']}"
        f"\nID: {f['id']}"
        f"\nSize: {f.get('size', 'N/A')}"
        f"\nType: {f['mimeType']}"
    )


Name: cache-3ea4c2f6881f03ff.arrow
ID: 11JZtmIsJo-8hDJSQw7Hn2Lnkmsh93V6U
Size: 553779336
Type: application/octet-stream

Name: cache-b7aeb6957386d36d.arrow
ID: 1ogprXBZaB-z0QthEm98ER8RmO1xTXQzR
Size: 520310224
Type: application/octet-stream

Name: cache-e57d4389b322bdc5.arrow
ID: 1NFjy4sta-0ElNmKK6gCTFgIYBp9Y6z6g
Size: 468409760
Type: application/octet-stream

Name: data-00000-of-00001.arrow
ID: 17pSUBzxRbVHBW_4RIWivVYBCxApwNb33
Size: 233086600
Type: application/octet-stream

Name: dataset_info.json
ID: 1_N6H8SZbbfOOyZwW3ybxVZ4JD2oSDw6W
Size: 414
Type: application/json

Name: state.json
ID: 1tkwi1-LGlXQwbnmDGBG9VFNeWmww6rLe
Size: 247
Type: application/json


In [ ]:
!pip install -q google-auth google-auth-oauthlib google-api-python-client

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving client_secret_1002707684834-d13oelj6tgul4fntbba5ooe1bt6hl1nj.apps.googleusercontent.com.json to client_secret_1002707684834-d13oelj6tgul4fntbba5ooe1bt6hl1nj.apps.googleusercontent.com.json


In [ ]:
import os

print(os.listdir("."))

['notebooks', 'src', 'models', 'datasets', 'outputs', 'configs', 'environment', 'logs', 'tests', '.env.example', '.gitignore', 'README.md', 'data', 'checkpoints', 'scripts', 'results', 'client_secret_1002707684834-d13oelj6tgul4fntbba5ooe1bt6hl1nj.apps.googleusercontent.com.json']


In [ ]:
import os

files = [f for f in os.listdir(".") if f.endswith(".json")]
print(files)

['client_secret_1002707684834-d13oelj6tgul4fntbba5ooe1bt6hl1nj.apps.googleusercontent.com.json']


In [ ]:
import os

old_name = [f for f in os.listdir(".") if f.startswith("client_secret_")][0]

os.rename(old_name, "credentials.json")

print("Renamed successfully:", os.path.exists("credentials.json"))

Renamed successfully: True


In [ ]:
from google.colab import auth
auth.authenticate_user()

print("Google authentication successful!")

Google authentication successful!


In [ ]:
from googleapiclient.discovery import build
from google.auth import default

SCOPES = [
    "https://www.googleapis.com/auth/drive"
]

creds, _ = default(scopes=SCOPES)

service = build(
    "drive",
    "v3",
    credentials=creds
)

result = service.files().list(
    pageSize=10,
    fields="files(id,name,mimeType)"
).execute()

for f in result.get("files", []):
    print(
        f"{f['name']} | "
        f"{f['mimeType']} | "
        f"{f['id']}"
    )

credentials.json | application/json | 1WKOY7lSU4IyFBl1dOQN1XslfUuUODQ9X
Judging_LLMs_.ipynb | application/vnd.google.colaboratory | 1lz2xS94o-lj5G7_9kUywklfIg5T3K7nT
Anmol_Backend_resume.pdf | application/pdf | 1CymQHDtOAopCGQsavUvaYaXhOOzsQIN6
LLM-Judge | application/vnd.google-apps.folder | 17dMO0vsTcqzBc2EGLLTxM9ZH6AgB50SU
cache-e57d4389b322bdc5.arrow | application/octet-stream | 1NFjy4sta-0ElNmKK6gCTFgIYBp9Y6z6g
data-00000-of-00001.arrow | application/octet-stream | 1gjbFzT0qmLRFEPTPuHJ-jinbz68Cju7N
dataset_info.json | application/json | 1_N6H8SZbbfOOyZwW3ybxVZ4JD2oSDw6W
dataset_info.json | application/json | 1_yDm3nXgXmpkLQ_gN_gQuRoybLO8DfnF
state.json | application/json | 16BJo3UBfqMF-z99dcvxHC-5zcTogiSqw
state.json | application/json | 1tkwi1-LGlXQwbnmDGBG9VFNeWmww6rLe


In [ ]:
from googleapiclient.discovery import build
from google.auth import default

SCOPES = [
    "https://www.googleapis.com/auth/drive"
]

creds, _ = default(scopes=SCOPES)

service = build(
    "drive",
    "v3",
    credentials=creds
)

result = service.files().list(
    pageSize=10,
    fields="files(id,name,mimeType)"
).execute()

for f in result.get("files", []):
    print(
        f"{f['name']} | "
        f"{f['mimeType']} | "
        f"{f['id']}"
    )

credentials.json | application/json | 1WKOY7lSU4IyFBl1dOQN1XslfUuUODQ9X
Judging_LLMs_.ipynb | application/vnd.google.colaboratory | 1lz2xS94o-lj5G7_9kUywklfIg5T3K7nT
Anmol_Backend_resume.pdf | application/pdf | 1CymQHDtOAopCGQsavUvaYaXhOOzsQIN6
LLM-Judge | application/vnd.google-apps.folder | 17dMO0vsTcqzBc2EGLLTxM9ZH6AgB50SU
cache-e57d4389b322bdc5.arrow | application/octet-stream | 1NFjy4sta-0ElNmKK6gCTFgIYBp9Y6z6g
data-00000-of-00001.arrow | application/octet-stream | 1gjbFzT0qmLRFEPTPuHJ-jinbz68Cju7N
dataset_info.json | application/json | 1_N6H8SZbbfOOyZwW3ybxVZ4JD2oSDw6W
dataset_info.json | application/json | 1_yDm3nXgXmpkLQ_gN_gQuRoybLO8DfnF
state.json | application/json | 16BJo3UBfqMF-z99dcvxHC-5zcTogiSqw
state.json | application/json | 1tkwi1-LGlXQwbnmDGBG9VFNeWmww6rLe


In [ ]:
results = service.files().list(
    q="name = 'LLM-Judge' and trashed = false",
    fields="files(id,name,mimeType,parents)",
    pageSize=20,
).execute()

for f in results.get("files", []):
    print(f)

{'mimeType': 'application/vnd.google-apps.folder', 'parents': ['0AAF3oWR6BWuOUk9PVA'], 'id': '17dMO0vsTcqzBc2EGLLTxM9ZH6AgB50SU', 'name': 'LLM-Judge'}


In [ ]:
from googleapiclient.discovery import build
from google.auth import default

SCOPES = ["https://www.googleapis.com/auth/drive"]

creds, _ = default(scopes=SCOPES)

service = build(
    "drive",
    "v3",
    credentials=creds
)

result = service.files().list(
    pageSize=10,
    fields="files(id,name,mimeType)"
).execute()

print("Google Drive API connected successfully!\n")

for f in result.get("files", []):
    print(f["name"], "|", f["id"])

Google Drive API connected successfully!

credentials.json | 1WKOY7lSU4IyFBl1dOQN1XslfUuUODQ9X
Judging_LLMs_.ipynb | 1lz2xS94o-lj5G7_9kUywklfIg5T3K7nT
Anmol_Backend_resume.pdf | 1CymQHDtOAopCGQsavUvaYaXhOOzsQIN6
LLM-Judge | 17dMO0vsTcqzBc2EGLLTxM9ZH6AgB50SU
cache-e57d4389b322bdc5.arrow | 1NFjy4sta-0ElNmKK6gCTFgIYBp9Y6z6g
data-00000-of-00001.arrow | 1gjbFzT0qmLRFEPTPuHJ-jinbz68Cju7N
dataset_info.json | 1_N6H8SZbbfOOyZwW3ybxVZ4JD2oSDw6W
dataset_info.json | 1_yDm3nXgXmpkLQ_gN_gQuRoybLO8DfnF
state.json | 16BJo3UBfqMF-z99dcvxHC-5zcTogiSqw
state.json | 1tkwi1-LGlXQwbnmDGBG9VFNeWmww6rLe


In [ ]:
results = service.files().list(
    q="name = 'LLM-Judge' and trashed = false",
    fields="files(id,name,mimeType,parents)",
    pageSize=20
).execute()

files = results.get("files", [])

if files:
    for f in files:
        print("LLM-Judge folder found:")
        print("Name :", f["name"])
        print("ID   :", f["id"])
else:
    print("LLM-Judge folder not found.")

LLM-Judge folder found:
Name : LLM-Judge
ID   : 17dMO0vsTcqzBc2EGLLTxM9ZH6AgB50SU


In [ ]:
from googleapiclient.discovery import build
from google.auth import default

SCOPES = ["https://www.googleapis.com/auth/drive"]

creds, _ = default(scopes=SCOPES)
service = build("drive", "v3", credentials=creds)

ROOT_ID = "17dMO0vsTcqzBc2EGLLTxM9ZH6AgB50SU"


def get_or_create_folder(name, parent_id):
    query = (
        f"name = '{name}' "
        f"and '{parent_id}' in parents "
        f"and mimeType = 'application/vnd.google-apps.folder' "
        f"and trashed = false"
    )

    result = service.files().list(
        q=query,
        fields="files(id,name)",
        pageSize=10
    ).execute()

    files = result.get("files", [])

    if files:
        return files[0]["id"]

    metadata = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id]
    }

    folder = service.files().create(
        body=metadata,
        fields="id,name"
    ).execute()

    return folder["id"]


# Create:
# LLM-Judge/
#   outputs/
#     kaggle/
#       stage1/

outputs_id = get_or_create_folder("outputs", ROOT_ID)
kaggle_id = get_or_create_folder("kaggle", outputs_id)
stage1_id = get_or_create_folder("stage1", kaggle_id)

print("Checkpoint structure ready!")
print("outputs :", outputs_id)
print("kaggle  :", kaggle_id)
print("stage1  :", stage1_id)

Checkpoint structure ready!
outputs : 1UXWA2SeAUjqjOSSRDMD37o3rLG_hrbHA
kaggle  : 1zSJ5rEG6WKDz8d4IiTRzhytQdMQNXCLy
stage1  : 1rtwK-QEMRsQ8oZhNt07HBGbZHYincCPK


In [ ]:
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build
from google.auth import default
import os
import json
import time

SCOPES = ["https://www.googleapis.com/auth/drive"]

creds, _ = default(scopes=SCOPES)
service = build("drive", "v3", credentials=creds)

STAGE1_FOLDER_ID = "1rtwK-QEMRsQ8oZhNt07HBGbZHYincCPK"


def create_drive_folder(name, parent_id):
    metadata = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id]
    }

    folder = service.files().create(
        body=metadata,
        fields="id,name"
    ).execute()

    return folder["id"]


def upload_file_to_drive(local_path, parent_id, drive_name=None):

    if not os.path.exists(local_path):
        raise FileNotFoundError(local_path)

    if drive_name is None:
        drive_name = os.path.basename(local_path)

    metadata = {
        "name": drive_name,
        "parents": [parent_id]
    }

    media = MediaFileUpload(
        local_path,
        resumable=True
    )

    request = service.files().create(
        body=metadata,
        media_body=media,
        fields="id,name,size"
    )

    response = None

    while response is None:
        try:
            status, response = request.next_chunk()

            if status:
                print(
                    f"Uploading {drive_name}: "
                    f"{int(status.progress() * 100)}%"
                )

        except Exception as e:
            print(f"Upload interrupted: {e}")
            print("Retrying...")
            time.sleep(5)

    print(f"Uploaded: {drive_name}")

    return response


def upload_checkpoint(checkpoint_path, global_step):

    checkpoint_name = f"checkpoint-{global_step}"

    print("=" * 60)
    print(f"Uploading {checkpoint_name}")
    print("=" * 60)

    # Create checkpoint folder
    checkpoint_folder_id = create_drive_folder(
        checkpoint_name,
        STAGE1_FOLDER_ID
    )

    uploaded_files = []

    # Upload every file in checkpoint
    for root, _, files in os.walk(checkpoint_path):

        for filename in files:

            local_file = os.path.join(root, filename)

            relative_path = os.path.relpath(
                local_file,
                checkpoint_path
            )

            drive_name = relative_path.replace(
                os.sep,
                "__"
            )

            upload_file_to_drive(
                local_file,
                checkpoint_folder_id,
                drive_name
            )

            uploaded_files.append(relative_path)

    # Manifest
    manifest = {
        "checkpoint": checkpoint_name,
        "global_step": global_step,
        "files": uploaded_files,
        "timestamp": time.time()
    }

    manifest_path = "/tmp/checkpoint_manifest.json"

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    upload_file_to_drive(
        manifest_path,
        checkpoint_folder_id,
        "manifest.json"
    )

    # IMPORTANT:
    # _COMPLETE is created ONLY after every file succeeds.
    marker_path = "/tmp/_COMPLETE"

    with open(marker_path, "w") as f:
        f.write("COMPLETE\n")

    upload_file_to_drive(
        marker_path,
        checkpoint_folder_id,
        "_COMPLETE"
    )

    print()
    print(f"✅ {checkpoint_name} safely uploaded to Drive")
    print()

In [ ]:
import os

print(os.path.exists("credentials.json"))

True


In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = ["https://www.googleapis.com/auth/drive"]

flow = InstalledAppFlow.from_client_secrets_file(
    "credentials.json",
    SCOPES
)

flow.redirect_uri = "urn:ietf:wg:oauth:2.0:oob"

auth_url, _ = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

print("OPEN THIS URL IN YOUR BROWSER:")
print(auth_url)

OPEN THIS URL IN YOUR BROWSER:
https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1002707684834-d13oelj6tgul4fntbba5ooe1bt6hl1nj.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=96XJGyXOFb0E6Rp8L20qzlNJeQHRv9&code_challenge=rLlcGw4mdXEfviknZXtUbSPo5T89twHulGeYCnwfDdU&code_challenge_method=S256&access_type=offline&prompt=consent


In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = ["https://www.googleapis.com/auth/drive"]

flow = InstalledAppFlow.from_client_secrets_file(
    "credentials.json",
    SCOPES
)

flow.redirect_uri = "urn:ietf:wg:oauth:2.0:oob"

auth_url, _ = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

print(auth_url)

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1002707684834-d13oelj6tgul4fntbba5ooe1bt6hl1nj.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=jkBMNRseh0cYG3pVoWOwAntc4EAAKF&code_challenge=OgNtNL7h10UwRBhxwCq2rzTrbnD_pXK80vyT1GWFsNw&code_challenge_method=S256&access_type=offline&prompt=consent


In [ ]:
code = input("Paste authorization code here: ")
flow.fetch_token(code=code)

creds = flow.credentials

print("Authentication successful!")
print("Refresh token generated:", creds.refresh_token is not None)

Paste authorization code here: 4/1ATsMZqCHQgSd_IdCsnvtDyaimYmAtXPwVyMOcWAHQczAjKbWYwKmlea3Zf0
Authentication successful!
Refresh token generated: True


In [ ]:
creds = flow.credentials

print("Authentication successful!")
print("Refresh token generated:", creds.refresh_token is not None)

Authentication successful!
Refresh token generated: True


In [ ]:
# Sanitized: Output containing credentials has been cleared.
print('Credentials successfully sanitized and cleared.')

# benchmark

In [ ]:
!nvidia-smi

Wed Sep 16 18:25:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/LLM-Judge"

for root, dirs, files in os.walk(BASE_DIR):
    if "adapter_model.safetensors" in files:
        print("FOUND:", root)

FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-250
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-500
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-750
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1000
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1250
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1500
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1563
FOUND: /content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/final


In [ ]:
import os
import torch

BASE_MODEL = "/content/drive/MyDrive/LLM-Judge/models/Qwen2.5-7B-Instruct"
FINAL_ADAPTER = "/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/final"

print("GPU available:", torch.cuda.is_available())
print("Base model exists:", os.path.exists(BASE_MODEL))
print("Final adapter exists:", os.path.exists(FINAL_ADAPTER))

print("\nFinal adapter files:")
for f in sorted(os.listdir(FINAL_ADAPTER)):
    print(" -", f)

GPU available: True
Base model exists: True
Final adapter exists: True

Final adapter files:
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - added_tokens.json
 - chat_template.jinja
 - merges.txt
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin
 - vocab.json


In [ ]:
import os

SEARCH_DIRS = [
    "/content/drive/MyDrive/LLM-Judge",
    "/kaggle/working"
]

keywords = [
    "benchmark",
    "evaluation",
    "eval",
    "hard",
    "technical"
]

for base in SEARCH_DIRS:
    print(f"\n{'='*70}")
    print(f"SEARCHING: {base}")
    print(f"{'='*70}")

    for root, dirs, files in os.walk(base):
        # Skip large/unnecessary directories
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__", "node_modules"]
        ]

        for f in files:
            name = f.lower()
            if any(k in name for k in keywords):
                print(os.path.join(root, f))


SEARCHING: /content/drive/MyDrive/LLM-Judge
/content/drive/MyDrive/LLM-Judge/src/judge/evaluator.py
/content/drive/MyDrive/LLM-Judge/src/evaluation/benchmark.py
/content/drive/MyDrive/LLM-Judge/src/evaluation/technical_benchmark.py
/content/drive/MyDrive/LLM-Judge/src/evaluation/hard_benchmark.py
/content/drive/MyDrive/LLM-Judge/datasets/benchmark/judge_benchmark.json
/content/drive/MyDrive/LLM-Judge/datasets/benchmark/position_bias_benchmark.json
/content/drive/MyDrive/LLM-Judge/datasets/benchmark/technical_50.json
/content/drive/MyDrive/LLM-Judge/datasets/benchmark/hard_technical_100.json
/content/drive/MyDrive/LLM-Judge/outputs/benchmarks/technical_50_results.json
/content/drive/MyDrive/LLM-Judge/outputs/benchmarks/hard_technical_100_results.json

SEARCHING: /kaggle/working


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/LLM-Judge")

files = [
    BASE / "src/evaluation/technical_benchmark.py",
    BASE / "src/evaluation/hard_benchmark.py",
    BASE / "src/evaluation/benchmark.py",
    BASE / "src/judge/evaluator.py",
]

for path in files:
    print("\n" + "=" * 90)
    print(f"FILE: {path}")
    print("=" * 90)

    if path.exists():
        print(path.read_text())
    else:
        print("FILE NOT FOUND")


FILE: /content/drive/MyDrive/LLM-Judge/src/evaluation/technical_benchmark.py
import json
import time
from pathlib import Path


class TechnicalBenchmark:

    def __init__(self, judge, dataset_path):
        self.judge = judge
        self.dataset_path = Path(dataset_path)

    def load_dataset(self):

        with open(
            self.dataset_path,
            "r",
            encoding="utf-8"
        ) as file:
            return json.load(file)

    def evaluate_pair(
        self,
        item,
        answer_a,
        answer_b,
        expected
    ):

        start_time = time.perf_counter()

        evaluation = self.judge.evaluate(
            problem=item["problem"],
            answer_a=answer_a,
            answer_b=answer_b
        )

        latency = (
            time.perf_counter()
            - start_time
        )

        winner = evaluation["result"]["winner"]

        return {
            "winner": winner,
            "expected": expected,
            "correct"

In [ ]:
import sys
import os
import torch

# Project root
PROJECT_ROOT = "/content/drive/MyDrive/LLM-Judge"
sys.path.insert(0, PROJECT_ROOT)

BASE_MODEL = f"{PROJECT_ROOT}/models/Qwen2.5-7B-Instruct"
FINAL_ADAPTER = f"{PROJECT_ROOT}/outputs/kaggle/stage1/final"

print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

print("Loading base Qwen model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    FINAL_ADAPTER,
)

model.eval()

print("\n" + "=" * 70)
print("FINE-TUNED MODEL LOADED")
print("=" * 70)
print("Base:", BASE_MODEL)
print("Adapter:", FINAL_ADAPTER)
print("Device:", model.device)

CUDA: True
GPUs: 1

Loading tokenizer...
Loading base Qwen model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading LoRA adapter...


ValueError: We need an `offload_dir` to dispatch this model according to this `device_map`, the following submodules need to be offloaded: base_model.model.model.layers.13, base_model.model.model.layers.14, base_model.model.model.layers.15, base_model.model.model.layers.16, base_model.model.model.layers.17, base_model.model.model.layers.18, base_model.model.model.layers.19, base_model.model.model.layers.20, base_model.model.model.layers.21, base_model.model.model.layers.22, base_model.model.model.layers.23, base_model.model.model.layers.24, base_model.model.model.layers.25, base_model.model.model.layers.26, base_model.model.model.layers.27, base_model.model.model.norm, base_model.model.model.rotary_emb, base_model.model.lm_head.

# Hugging face pushing

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive"

print("Searching for adapter_config.json...\n")

matches = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    if "adapter_config.json" in files:
        matches.append(os.path.join(root, "adapter_config.json"))

if matches:
    print("✓ Adapter found:")
    for path in matches:
        print(path)
else:
    print("❌ adapter_config.json not found")

Searching for adapter_config.json...

✓ Adapter found:
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-250/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-500/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-750/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1000/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1250/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1500/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/checkpoint-1563/adapter_config.json
/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/final/adapter_config.json


In [ ]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 20.0 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

login()

In [ ]:
import json
import os

ADAPTER_DIR = "/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/final"

CONFIG_PATH = os.path.join(
    ADAPTER_DIR,
    "adapter_config.json"
)

# Read adapter config
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

print("Old base_model:")
print(config.get("base_model_name_or_path"))

# Replace local Kaggle path with the actual HF model ID
config["base_model_name_or_path"] = "Qwen/Qwen2.5-7B-Instruct"

# Save corrected config
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\n✓ adapter_config.json corrected")
print("New base_model:")
print(config["base_model_name_or_path"])

Old base_model:
/kaggle/working/Qwen2.5-7B-Instruct

✓ adapter_config.json corrected
New base_model:
Qwen/Qwen2.5-7B-Instruct


In [ ]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": "Qwen/Qwen2.5-7B-Instruct",
  "bias": "none",
  "corda_config": null,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "qalora_group_size": 16,
  "r": 8,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "o_proj",
    "k_proj",
    "q_proj",
    "v_proj"
  ],
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trainable_token_indices": null,
  "use_dora": false,
  "use_qalora": false,
  "use_rslora": false
}


In [ ]:
import os

ADAPTER_DIR = "/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/final"
README_PATH = os.path.join(ADAPTER_DIR, "README.md")

print("README exists:", os.path.exists(README_PATH))

if os.path.exists(README_PATH):

    with open(README_PATH, "r", encoding="utf-8") as f:
        content = f.read()

    print("\nOld path found:", "/kaggle/working/Qwen2.5-7B-Instruct" in content)

    content = content.replace(
        "/kaggle/working/Qwen2.5-7B-Instruct",
        "Qwen/Qwen2.5-7B-Instruct"
    )

    with open(README_PATH, "w", encoding="utf-8") as f:
        f.write(content)

    print("✓ README.md corrected")

    print("\nRemaining old path:",
          "/kaggle/working/Qwen2.5-7B-Instruct" in content)

else:
    print("No README.md found.")

README exists: True

Old path found: True
✓ README.md corrected

Remaining old path: False


In [ ]:
from huggingface_hub import upload_folder

ADAPTER_DIR = "/content/drive/MyDrive/LLM-Judge/outputs/kaggle/stage1/final"

upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id="Anmol2507/LLM_as_judge_fine_tuned_Qwen_7B",
    repo_type="model",
    commit_message="Upload final LoRA adapter"
)

print("✓ FINAL ADAPTER UPLOADED SUCCESSFULLY")

✓ FINAL ADAPTER UPLOADED SUCCESSFULLY
